# EGVWC-KAN v2, Stage 2: confirmation on new datasets (shard A)

This notebook runs EGVW-KAN v1, ablation rows A to K and the frozen v2-final rows on the 13 Stage 2 datasets. None of them was used in the v1 benchmark or in Stage 1. Their corrections and the inclusion audit (rules R1-R5, as in v1) were fixed in the amendment to `preregistration_v2.md` before any model was trained on them. v2-final is the definition frozen after Stage 1 (gamma = 1, C = 100, gate on).

The three Stage 2 notebooks build the same folds from `BASE_SEED`, and their outputs merge in the analysis. `PIPELINE_VERSION` differs from v1 and Stage 1, so their results cannot be mixed with older runs. The first session needs no input; later sessions attach this notebook's own output to resume.

## Cell 1: install packages and clone the FC-KAN repository

In [ ]:
# Cell 1: install packages and clone FC-KAN (once per Kaggle session, Internet must be on)
# Installs the libraries this notebook imports. Packages that ship with the Kaggle image
# (numpy, pandas, torch, ...) are only checked, not reinstalled.
# A missing library needed by every part raises an error. A library needed by one part only gives a warning;
# the self-test in Stage 2 stops that part later.
import importlib, os, shutil, subprocess, sys
# Set before the first torch import. MKL_CBWR fixes the Intel MKL code path so the same seed gives the same
# numbers on every run and CPU. Worker processes inherit the setting.
os.environ.setdefault("MKL_CBWR", "COMPATIBLE")
os.environ.setdefault("KMP_DETERMINISTIC_REDUCTION", "yes")

def _pip(pkg):
    args = [pkg]
    if pkg == "imbalanced-learn":  # keep the installed scikit-learn: a newer version could change the folds
        import sklearn; args.append(f"scikit-learn=={sklearn.__version__}")
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *args],
                       capture_output=True, text=True)
    return r.returncode == 0

# (import name, pip name, used by). Order matters: pykan does not declare its dependencies,
# so tqdm, sympy and pyyaml have to be installed before `import kan`.
REQUIRED = [
    ("numpy",        "numpy",        "all"),
    ("pandas",       "pandas",       "all"),
    ("scipy",        "scipy",        "all"),
    ("sklearn",      "scikit-learn", "all"),
    ("matplotlib",   "matplotlib",   "all"),
    ("joblib",       "joblib",       "all"),
    ("ucimlrepo",    "ucimlrepo",    "all"),  # downloads
    ("openpyxl",     "openpyxl",     "all"),  # .xlsx (Cryotherapy, Immunotherapy)
    ("xlrd",         "xlrd",         "all"),  # .xls (Breast Tissue)
    ("torch",        "torch",        "all"),
    ("tqdm",         "tqdm",         "Part 1"),  # needed by pykan
    ("sympy",        "sympy",        "Part 1"),  # needed by pykan
    ("yaml",         "pyyaml",       "Part 1"),  # needed by pykan
    ("kan",          "pykan",        "Part 1"),
    ("xgboost",      "xgboost",      "Part 2"),
    ("lightgbm",     "lightgbm",     "Part 2"),
    ("catboost",     "catboost",     "Part 2"),
    ("transformers", "transformers", "Part 3"),  # needed by faster_kan.py in FC-KAN
    ("imblearn",     "imbalanced-learn", "Part 4"),  # SMOTE-NC; installed against the existing scikit-learn
    ("baycomp",      "baycomp",      "optional"),  # Bayesian signed-rank test (skipped if missing)
]
status, failed = {}, []
for imp, pipname, needed_by in REQUIRED:
    try:
        importlib.import_module(imp); status[pipname] = "present"
    except Exception:
        ok = _pip(pipname)
        importlib.invalidate_caches()
        try:
            importlib.import_module(imp); status[pipname] = "installed"
        except Exception as e:
            status[pipname] = f"FAILED ({type(e).__name__}: {str(e)[:60]})"
            failed.append((pipname, needed_by))
    print(f"  {pipname:<14} {status[pipname]:<45} needed by: {needed_by}")

# FC-KAN repository: code of the eight KAN variants
FCKAN_DIR = os.path.abspath("./github_kan_repos/FC_KAN")
if not os.path.isdir(os.path.join(FCKAN_DIR, "models")):
    if shutil.which("git"):
        os.makedirs(os.path.dirname(FCKAN_DIR), exist_ok=True)
        for attempt in range(3):
            subprocess.run(["git", "clone", "--depth", "1", "https://github.com/hoangthangta/FC_KAN", FCKAN_DIR],
                           capture_output=True, text=True)
            if os.path.isdir(os.path.join(FCKAN_DIR, "models")): break
            shutil.rmtree(FCKAN_DIR, ignore_errors=True)
repo_ok = os.path.isdir(os.path.join(FCKAN_DIR, "models"))
print(f"  {'FC_KAN repo':<14} {'OK' if repo_ok else 'MISSING (git clone failed)':<45} needed by: Part 3")
if not repo_ok: failed.append(("FC_KAN repository", "Part 3"))

core = [p for p, n in failed if n == "all"]
if core:
    raise RuntimeError(f"Required by every part and could not be installed: {core}. "
                       f"Check that Internet is ON in the notebook settings, then run this cell again.")
for p, n in failed:
    if n != "optional":
        print(f"  WARNING: {p} is missing - {n} cannot run until it is installed (its Stage 2 self-test will stop).")
print("  Install check finished.")

## Cell 2: configuration
`NOTEBOOK` selects the part this notebook runs. Only `SMOKE` should be edited, and in all parts at once.

In [ ]:
# Cell 2: configuration. Normally the only cell to edit.
import os, time

# Part run by this notebook: "kan" (EGVW-KAN, tuned single KAN, ablation rows),
# "baselines" (8 classical models), "variants" (8 KAN variants) or "resampling" (Part 4).
# All notebooks build the same folds from BASE_SEED.
NOTEBOOK = "kan"
# Half of the datasets run here: "A" or "B" (see SHARD_A_DATASETS).
# We ran the "_A" notebooks on one Kaggle account and the "_B" notebooks on another.
SHARD = "A"
_ROOT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

CONFIG = dict(
    # output and resume
    NOTEBOOK          = NOTEBOOK,
    PIPELINE_VERSION  = "2026-10-09-tahap2",  # Stage 2 results are never merged with v1 or Stage 1 results
    METHOD_VERSION    = "v2-single-correction",  # version of the EGVW-KAN definition; KAN and Part 4 units of other versions are
                                     # not merged. Not part of the fingerprint, so baseline and variant results stay mergeable.
    VARIANT_VERSION   = "v2-no-input-layernorm",  # version of the KAN-variant changes in Cell 7; variant units of other
                                     # versions are not merged.
    SAVE_DIR          = os.path.join(_ROOT, f"egvw_results_{NOTEBOOK}_{SHARD}"),
    # earlier outputs of this and the other notebooks, attached with Add Input > Your Work
    RESUME_GLOB       = "/kaggle/input/**/egvw_results_*",
    MAX_RUNTIME_HOURS = 10.5,  # a new unit starts only if it should finish before this limit
    HARD_STOP_HOURS   = 11.25,  # running units are stopped here (Kaggle limit is 12 h)
    SESSION_START     = time.time(),
    UNSEEN_UNIT_GUESS_HOURS = 2.0,  # assumed duration of unit types without a timing yet
    N_JOBS            = max(1, (os.cpu_count() or 2)),

    # protocol
    REPEATS        = 3,  # repeated nested CV: R x K outer folds
    OUTER_FOLDS    = 5,
    INNER_FOLDS    = 3,
    BASE_SEED      = 2026,
    N_CANDIDATES   = 12,  # same tuning budget for every tuned method
    SELECTION_METRIC = "mcc",  # selection metric of the inner CV

    # EGVW-KAN settings
    N_MEMBERS      = 5,
    GAMMA          = 1.0,
    INNER_VAL_FRAC = 0.2,  # internal validation split, taken from the training fold only
    REFIT          = True,  # refit members on all training rows once the weights are fixed
    IR_THRESHOLDS  = (1.5, 3.0, 10.0),
    TIER_EXPONENT  = {"balanced": 0.0, "mild": 0.5, "moderate": 1.0, "extreme": 1.5},
    WEIGHT_CAP     = 10.0,  # class weights capped at 10 x median
    OVERFIT_GAP    = 0.05, OVERFIT_RATIO = 1.5, OVERFIT_LAMB_MULT = 5.0,
    UNDERFIT_MARGIN = 0.90, UNDERFIT_STEPS_MULT = 2, UNDERFIT_STEPS_CAP = 100,
    KAN_GRID = {  # 3 x 2 x 1 x 3 x 2 x 2 = 72 combinations, 12 sampled
        "n_hidden": [3, 5, 7], "grid": [3, 5], "k": [3],
        "lr": [0.005, 0.01, 0.05], "lamb": [0.001, 0.01], "steps": [30, 50],
    },

    # what to run (follows NOTEBOOK)
    RUN_KAN        = NOTEBOOK in ("kan", "all"),  # tuned single KAN, EGVW-KAN and ablation rows
    RUN_ABLATION   = True,  # ladder rows B-D and I (A, E-H and J come for free)
    RUN_BASELINES  = NOTEBOOK in ("baselines", "all"),  # LR, SVM, KNN, RF, XGBoost, LightGBM, CatBoost, MLP
    RUN_VARIANTS   = NOTEBOOK in ("variants", "all"),  # the eight variants from the FC-KAN repository
    RUN_RESAMPLING = NOTEBOOK == "resampling",  # Part 4: SMOTE-NC, undersampling, threshold moving
    RUN_V2         = False,  # Stage 2: v2-final rows are computed inside each KAN unit (Cell 7d)
    # EGVWC-KAN v2 (Part 5, see preregistration_v2.md). Not in the fingerprint, so v2 pairs with Part 1.
    V2_VERSION     = "v2.0-final",  # frozen after Stage 1 (defaults kept)
    V2_C_SHRINK    = 100,  # gamma_eff = gamma * n_oob / (n_oob + C)
    V2_GAMMA_GRID  = [0.5, 1.0, 2.0, 4.0],  # post-hoc sensitivity rows (no retraining)
    V2_SHRINK_GRID = [0, 25, 100, 400],
    GAMMA_GRID     = [0.25, 0.5, 1.0, 2.0, 4.0],  # post-hoc sensitivity rows (no retraining)

    # KAN variants: trainer settings
    VARIANT_EPOCHS   = 300,  # full-batch AdamW, cosine schedule, early stopping
    VARIANT_PATIENCE = 30,

    # dataset inclusion rules, fixed before training
    MIN_CLASS_COUNT       = 10,  # R1
    MIN_SAMPLES           = 40,  # R2
    MIN_FEATURES          = 3,  # R2
    COL_MISSING_DROP      = 0.40,  # column filter per fold (also used for R2)
    STATLOG_OVERLAP_MAX   = 0.50,  # R4: Statlog is dropped if more than 50% of its rows are in Cleveland
    BUPA_DRINKS_THRESHOLD = 3,  # BUPA target: drinks > 3 (McDermott and Forsyth, PRL 2016)
    DATASETS = None,  # None = all candidates that pass the rules, or a list of names

    # Split over two Kaggle accounts. Not in the protocol fingerprint: folds and seeds do not depend on it,
    # so the A and B results merge as if one notebook had run everything.
    # The list is fixed (balanced by rows x features) and is not rebuilt from what downloaded.
    # Change it in all notebooks or not at all.
    SHARD = SHARD,
    SHARD_A_DATASETS = ["AIDS Clinical Trials (ACTG 175)", "Cervical Cancer Risk Factors", "Diabetic Retinopathy Debrecen", "HCV Data", "Haberman's Survival", "Fertility", "Parkinson's (Little 2008)"],  # all other included datasets go to shard B

    # quick end-to-end check (a few minutes). True for a first run, then False.
    SMOKE = False,
)

if CONFIG["SMOKE"]:
    CONFIG.update(REPEATS=1, OUTER_FOLDS=2, INNER_FOLDS=2, N_CANDIDATES=2, N_MEMBERS=3,
                  VARIANT_EPOCHS=20, VARIANT_PATIENCE=5,
                  KAN_GRID={"n_hidden": [3], "grid": [3], "k": [3], "lr": [0.01],
                            "lamb": [0.001], "steps": [5, 8]},
                  DATASETS=CONFIG["DATASETS"] or ["Breast Cancer Wisconsin (Diagnostic)", "Maternal Health Risk"],
                  SAVE_DIR=CONFIG["SAVE_DIR"] + "_smoke")
os.makedirs(CONFIG["SAVE_DIR"], exist_ok=True)

# Protocol fingerprint. Results from another notebook or session are merged only when it matches,
# so smoke runs or runs with other folds never end up in the results.
PROTOCOL_KEYS = ["PIPELINE_VERSION", "REPEATS", "OUTER_FOLDS", "INNER_FOLDS", "BASE_SEED", "N_CANDIDATES", "SMOKE", "DATASETS",
                 "N_MEMBERS", "GAMMA", "INNER_VAL_FRAC", "BUPA_DRINKS_THRESHOLD", "COL_MISSING_DROP"]
import json as _json
_json.dump({k: CONFIG[k] for k in PROTOCOL_KEYS}, open(os.path.join(CONFIG["SAVE_DIR"], "protocol.json"), "w"),
           indent=1, default=str)
print(f"NOTEBOOK = {NOTEBOOK!r} -> runs: " + ", ".join(g for g, k in (("EGVW-KAN + ablation", "RUN_KAN"),
      ("classical baselines", "RUN_BASELINES"), ("KAN variants", "RUN_VARIANTS")) if CONFIG[k]))
print({k: v for k, v in CONFIG.items() if k not in ("KAN_GRID", "SESSION_START")})

## Cell 3: candidate datasets, corrections, inclusion rules R1-R5

In [ ]:
# Cell 3: candidate datasets, corrections and inclusion rules
# Each candidate is loaded, corrected and checked against rules R1-R5. The rules were
# fixed before training and none of them looks at model performance.
#   R1  every class has >= MIN_CLASS_COUNT samples
#   R2  >= MIN_SAMPLES samples and >= MIN_FEATURES usable features
#   R3  target is the clinical outcome documented by the donor (not an admin or split field)
#   R4  one dataset per patient cohort (Friedman tests need independent datasets)
#   R5  downloadable from its source at run time
# The decision for each candidate goes to dataset_inclusion.csv.
import json, os, time, warnings
from dataclasses import dataclass, field
from typing import Callable, List, Optional

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
MISSING_TOKENS = ["?", "NaNN", "NA", "N/A", "n/a", "NULL", "null", "", " ", "nan", "NaN", "\t?"]


@dataclass
class Bundle:
    name: str
    X: pd.DataFrame  # raw features with NaN; preprocessing is fitted per fold
    y: np.ndarray  # labels 0..K-1; in binary tasks 1 = clinical condition
    class_names: List[str]
    nominal: List[str]  # columns to one-hot encode
    groups: Optional[np.ndarray] = None  # patient id when a patient has several rows
    notes: List[str] = field(default_factory=list)

    @property
    def K(self): return len(self.class_names)
    @property
    def IR(self):
        c = np.bincount(self.y, minlength=self.K); c = c[c > 0]
        return float(c.max() / c.min())


# helpers
def _clean(df: pd.DataFrame) -> pd.DataFrame:
    """Normalise missing tokens; turn object columns that are really numeric into numbers."""
    df = df.copy()
    if df.columns.duplicated().any():  # e.g. Lymphography has two columns with the same name
        seen = {}
        cols = []
        for c in map(str, df.columns):
            seen[c] = seen.get(c, 0) + 1
            cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
        df.columns = cols
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):  # object or pandas 3 'str' dtype
            s = df[c].astype(str).str.strip().replace(MISSING_TOKENS, np.nan)
            num = pd.to_numeric(s, errors="coerce")
            df[c] = num if s.notna().sum() and num.notna().sum() >= 0.9 * s.notna().sum() else s
    return df


def _norm_label(v) -> str:
    if pd.isna(v): return np.nan
    if isinstance(v, (int, np.integer)): return str(int(v))
    if isinstance(v, (float, np.floating)): return str(int(v)) if float(v).is_integer() else str(v)
    return str(v).strip().lower()


def _encode_target(y_raw: pd.Series, positive=None, order=None):
    """Binary: positive = raw values meaning 'has the condition' -> 1.  Multiclass: explicit order."""
    y = y_raw.map(_norm_label)
    keep = y.notna().to_numpy()
    y = y[keep]
    if positive is not None:
        pos = {_norm_label(p) for p in positive}
        unknown = set(y.unique()) - pos
        if len(unknown) != 1:
            raise ValueError(f"expected exactly one negative class, found {sorted(set(y.unique()))}")
        return keep, (y.isin(pos)).astype(int).to_numpy(), [f"not {sorted(pos)[0]}", sorted(pos)[0]]
    order = [_norm_label(o) for o in order] if order else sorted(y.unique())
    missing = set(y.unique()) - set(order)
    if missing:
        raise ValueError(f"target values {missing} not in declared class order {order}")
    order = [o for o in order if o in set(y.unique())]
    return keep, y.map({c: i for i, c in enumerate(order)}).to_numpy().astype(int), order


def _drop_like(X: pd.DataFrame, names: List[str]) -> List[str]:
    """Drop columns whose lower-cased name matches any entry; return what was dropped."""
    low = {c.lower().replace("-", "_").replace(" ", "_"): c for c in X.columns}
    hit = [low[n.lower().replace("-", "_").replace(" ", "_")] for n in names
           if n.lower().replace("-", "_").replace(" ", "_") in low]
    X.drop(columns=hit, inplace=True)
    return hit


def _uci(uid):
    from ucimlrepo import fetch_ucirepo
    ds = fetch_ucirepo(id=uid)
    return ds.data.features.copy(), ds.data.targets.copy()


def _uci_zip(uid, slug):
    """Direct download for datasets the ucimlrepo API does not serve ('DatasetNotFoundError'):
    the zip on the dataset's UCI page, returning its first Excel/CSV sheet that has data."""
    import io, urllib.request, zipfile
    url = f"https://archive.ics.uci.edu/static/public/{uid}/{slug}.zip"
    with urllib.request.urlopen(url, timeout=60) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    for name in z.namelist():
        low = name.lower()
        if low.endswith((".xlsx", ".xls")):
            sheets = pd.read_excel(io.BytesIO(z.read(name)), sheet_name=None)
            best = max(sheets.values(), key=lambda s: s.shape[0] * s.shape[1])  # data sheet, not the notes sheet
            return best.dropna(how="all").dropna(axis=1, how="all")
        if low.endswith(".csv"):
            return pd.read_csv(io.BytesIO(z.read(name)))
    raise FileNotFoundError(f"no data file in {url}: {z.namelist()}")


def _col(df: pd.DataFrame, *keys) -> str:
    """First column whose lower-cased name contains any of the keys."""
    for k in keys:
        for c in df.columns:
            if k in str(c).lower(): return c
    raise KeyError(f"none of {keys} in columns {list(df.columns)}")


def _pick_target(T: pd.DataFrame, name: Optional[str]) -> pd.Series:
    if name is None: return T.iloc[:, 0]
    for c in T.columns:
        if c.lower().replace("-", "_") == name.lower().replace("-", "_"): return T[c]
    raise KeyError(f"target column '{name}' not among {list(T.columns)}")


# candidate registry
# Each entry: name, cohort id (R4), loader returning (X, y_raw, groups or None, notes), encoding.
CANDIDATES = []

def candidate(name, cohort, positive=None, order=None, nominal=(), r4_priority=0):
    def deco(fn):
        CANDIDATES.append(dict(name=name, cohort=cohort, positive=positive, order=order,
                               nominal=list(nominal), r4_priority=r4_priority, load=fn))
        return fn
    return deco


@candidate("Breast Cancer Wisconsin (Diagnostic)", "wisc_diag", positive=["malignant"])
def _():
    from sklearn.datasets import load_breast_cancer
    d = load_breast_cancer(as_frame=True)
    y = d.target.map({0: "malignant", 1: "benign"})  # sklearn codes malignant as 0
    return d.data, y, None, ["positive class = malignant (sklearn encodes it as 0)"]

@candidate("Breast Cancer Wisconsin (Original)", "wisc_orig", positive=["4"])
def _():
    X, T = _uci(15); n = _drop_like(X, ["Sample_code_number", "id"])
    return X, _pick_target(T, "Class"), None, [f"dropped {n}", "2 = benign, 4 = malignant"]

@candidate("Breast Cancer Wisconsin (Prognostic)", "wisc_prog", positive=["r"])
def _():
    X, T = _uci(16)
    n = _drop_like(X, ["ID", "Time"])  # Time is time to recurrence, derived from the outcome
    return X, _pick_target(T, "Outcome"), None, [f"dropped {n} (Time is outcome-derived)"]

@candidate("Breast Cancer Coimbra", "coimbra", positive=["2"])
def _():
    X, T = _uci(451); return X, _pick_target(T, "Classification"), None, ["2 = patient"]

@candidate("Mammographic Mass", "mammo", positive=["1"], nominal=["Shape", "Margin"])
def _():
    X, T = _uci(161)
    n = _drop_like(X, ["BI-RADS", "BI_RADS", "BIRADS"])  # BI-RADS is the radiologist's verdict; the donor marks it non-predictive
    return X, _pick_target(T, "Severity"), None, [f"dropped {n} (non-predictive per donor)"]

@candidate("Liver Disorders (BUPA)", "bupa", positive=["heavy"])
def _():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/liver-disorders/bupa.data"
    df = pd.read_csv(url, names=["mcv", "alkphos", "sgpt", "sgot", "gammagt", "drinks", "selector"])
    thr = CONFIG["BUPA_DRINKS_THRESHOLD"]
    y = (df["drinks"] > thr).map({True: "heavy", False: "light"})  # BUPA
    X = df.drop(columns=["drinks", "selector"])  # selector is the train/test flag
    return X, y, None, [f"target = drinks > {thr}; 'selector' (split flag) removed [McDermott & Forsyth 2016]"]

@candidate("Indian Liver Patient (ILPD)", "ilpd", positive=["1"], nominal=["Gender"])
def _():
    X, T = _uci(225); return X, _pick_target(T, "Selector"), None, ["1 = liver patient"]

@candidate("Parkinson's Telemonitoring", "parkinsons_tele", positive=["high"])
def _():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/parkinsons/telemonitoring/parkinsons_updrs.data"
    df = pd.read_csv(url)
    thr = float(df["total_UPDRS"].median())  # label defined once, from the outcome only
    y = (df["total_UPDRS"] > thr).map({True: "high", False: "low"})
    groups = df["subject#"].to_numpy()  # grouped splits: a patient is never on both sides
    X = df.drop(columns=["subject#", "age", "sex", "test_time", "motor_UPDRS", "total_UPDRS"])
    return X, y, groups, [f"grouped by subject (42 patients); label = total_UPDRS > {thr:.2f}; 16 voice features"]

@candidate("Pima Indians Diabetes", "pima", positive=["1"])
def _():
    url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
    cols = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI", "DiabetesPedigree", "Age", "Outcome"]
    df = pd.read_csv(url, names=cols)
    zero_is_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    df[zero_is_missing] = df[zero_is_missing].replace(0, np.nan)  # Pima zeros
    return df.drop(columns=["Outcome"]), df["Outcome"], None, ["physiologically impossible zeros -> missing"]

@candidate("Early-Stage Diabetes Risk", "esdr", positive=["positive"])
def _():
    X, T = _uci(529); return X, _pick_target(T, "class"), None, []

@candidate("Dermatology", "derma", order=["1", "2", "3", "4", "5", "6"])
def _():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data"
    df = pd.read_csv(url, header=None)
    df.columns = [f"f{i}" for i in range(df.shape[1] - 1)] + ["Class"]
    df["f33"] = pd.to_numeric(df["f33"].replace("?", np.nan), errors="coerce")  # age as a number
    return df.drop(columns=["Class"]), df["Class"], None, ["age parsed as numeric"]

CLEVELAND_COLS = ["age", "sex", "cp", "trestbps", "chol", "fbs", "restecg", "thalach",
                  "exang", "oldpeak", "slope", "ca", "thal"]

@candidate("Heart Disease (Cleveland)", "cleveland", positive=["disease"],
           nominal=["cp", "restecg", "slope", "thal"], r4_priority=1)
def _():
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
    df = pd.read_csv(url, names=CLEVELAND_COLS + ["num"], na_values="?")
    y = (df["num"] > 0).map({True: "disease", False: "no disease"})
    return df[CLEVELAND_COLS], y, None, ["target = num > 0"]

@candidate("Statlog (Heart)", "statlog", positive=["2"], nominal=["cp", "restecg", "slope", "thal"])
def _():
    X, T = _uci(145)
    X.columns = CLEVELAND_COLS[: X.shape[1]]  # same 13 attributes in the same order
    return X, T.iloc[:, 0], None, ["2 = presence"]

@candidate("SPECTF Heart", "spect", positive=["1"], r4_priority=1)
def _():
    X, T = _uci(96); return X, T.iloc[:, 0], None, ["original continuous ROI counts"]

@candidate("SPECT Heart", "spect", positive=["1"])
def _():
    X, T = _uci(95); return X, T.iloc[:, 0], None, ["binarised version of SPECTF"]

@candidate("Acute Inflammations (Bladder)", "acute", positive=["yes"], r4_priority=1)
def _():
    X, T = _uci(184); return X, _pick_target(T, "bladder-inflammation"), None, ["donor's first decision"]

@candidate("Acute Inflammations (Nephritis)", "acute", positive=["yes"])
def _():
    X, T = _uci(184); return X, _pick_target(T, "nephritis"), None, []

@candidate("Chronic Kidney Disease", "ckd", positive=["ckd"])
def _():
    X, T = _uci(336); return X, _pick_target(T, "class"), None, []

@candidate("Maternal Health Risk", "maternal", order=["low risk", "mid risk", "high risk"])
def _():
    X, T = _uci(863); return X, _pick_target(T, "RiskLevel"), None, []

@candidate("Differentiated Thyroid Cancer Recurrence", "thyroid", positive=["yes"])
def _():
    X, T = _uci(915); return X, _pick_target(T, "Recurred"), None, []

@candidate("Cardiotocography (NSP)", "ctg", order=["1", "2", "3"])
def _():
    X, T = _uci(193); n = _drop_like(X, ["CLASS", "NSP"])
    return X, _pick_target(T, "NSP"), None, ["R3: NSP (normal/suspect/pathologic) is the clinical fetal-state outcome", f"dropped {n}"]

@candidate("Hepatitis", "hepatitis", positive=["1"])
def _():
    X, T = _uci(46); return X, _pick_target(T, "Class"), None, ["1 = die"]

@candidate("Cirrhosis Patient Survival", "cirrhosis", order=["c", "cl", "d"])
def _():
    X, T = _uci(878)
    n = _drop_like(X, ["ID", "N_Days"])  # N_Days is follow-up until the outcome, so it is derived from the outcome
    return X, _pick_target(T, "Status"), None, [f"dropped {n} (N_Days is outcome-derived)"]

MI_LATER_DAY_COLS = ["R_AB_1_n", "R_AB_2_n", "R_AB_3_n", "NA_R_1_n", "NA_R_2_n", "NA_R_3_n",
                     "NOT_NA_1_n", "NOT_NA_2_n", "NOT_NA_3_n"]

@candidate("Myocardial Infarction Complications (AF)", "mi", positive=["1"])
def _():
    X, T = _uci(579)
    n = _drop_like(X, ["ID"] + MI_LATER_DAY_COLS)  # prediction at admission: no variables from later days
    return X, _pick_target(T, "FIBR_PREDS"), None, [f"target = atrial fibrillation; dropped {n}"]

# The next three are not in the ucimlrepo API, so we read the zip from their UCI page.
@candidate("Breast Tissue", "breast_tissue", order=["car", "fad", "mas", "gla", "con", "adi"])
def _():
    df = _uci_zip(192, "breast+tissue")
    t = _col(df, "class")
    X = df.drop(columns=[t] + [c for c in df.columns if "case" in str(c).lower()])
    return X, df[t], None, ["read from the UCI zip (not served by the ucimlrepo API)", "6 tissue classes"]

@candidate("Cryotherapy", "cryo", positive=["1"], nominal=["Type"])
def _():
    df = _uci_zip(429, "cryotherapy+dataset")
    t = _col(df, "result")
    return df.drop(columns=[t]), df[t], None, ["read from the UCI zip", "1 = wart cleared after cryotherapy"]

@candidate("Immunotherapy", "immuno", positive=["1"], nominal=["Type"])
def _():
    df = _uci_zip(428, "immunotherapy+dataset")
    t = _col(df, "result")
    return df.drop(columns=[t]), df[t], None, ["read from the UCI zip", "1 = wart cleared after immunotherapy"]

@candidate("Primary Tumor", "primary_tumor")
def _():
    X, T = _uci(83); return X, T.iloc[:, 0], None, []

@candidate("Lymphography", "lympho")
def _():
    X, T = _uci(63); return X, T.iloc[:, 0], None, []

@candidate("Audiology (Standardized)", "audiology")
def _():
    X, T = _uci(8); return X, T.iloc[:, 0], None, []


# audit: load, correct, apply the rules
def _statlog_overlap(stat: pd.DataFrame, clev: pd.DataFrame) -> float:
    a = stat[CLEVELAND_COLS].astype(float).round(1).fillna(-999)
    b = clev[CLEVELAND_COLS].astype(float).round(1).fillna(-999)
    keys_b = set(map(tuple, b.to_numpy()))
    return float(np.mean([tuple(r) in keys_b for r in a.to_numpy()]))


def audit_datasets(verbose=True):
    rows, bundles, loaded = [], {}, {}
    wanted = CONFIG.get("DATASETS")
    for c in CANDIDATES:
        if wanted and c["name"] not in wanted:
            continue
        rec = dict(dataset=c["name"], cohort=c["cohort"], decision="", reason="", n=None, d=None,
                   K=None, IR=None, duplicate_rows_grouped=None, missing_pct=None, notes="")
        try:
            for attempt in range(3):  # retry network errors so a dataset is not lost in one part only
                try:
                    X, y_raw, groups, notes = c["load"](); break
                except Exception as e:
                    permanent = "NotFound" in type(e).__name__ or isinstance(e, (KeyError, ValueError))
                    if attempt == 2 or permanent: raise  # retry network errors only
                    print(f"  {c['name']}: load failed ({type(e).__name__}); retrying in {15 * (attempt + 1)} s")
                    time.sleep(15 * (attempt + 1))
            X = _clean(X.reset_index(drop=True)); y_raw = y_raw.reset_index(drop=True)
            keep, y, classes = _encode_target(y_raw, c["positive"], c["order"])
            X = X.loc[keep].reset_index(drop=True)
            groups = None if groups is None else np.asarray(groups)[keep]
            # Identical records (same features and label) are kept, since with few coarse features they are
            # usually different patients. They are grouped, so they always land in the same fold and a
            # record is never on both sides of a split. Removing them would delete real patients and
            # change the class balance.
            full = pd.concat([X, pd.Series(y, name="__y")], axis=1)
            n_dup = int(full.duplicated().sum())
            if groups is None and n_dup:
                groups = pd.util.hash_pandas_object(full.astype(str), index=False).factorize()[0]
                notes = notes + [f"{n_dup} rows repeat an earlier row; kept, and identical rows share a fold"]
            nominal = [col for col in c["nominal"] if col in X.columns]
            cnt = np.bincount(y); cnt = cnt[cnt > 0]
            rec.update(n=len(y), d=X.shape[1], K=len(classes), IR=round(float(cnt.max() / cnt.min()), 2),
                       duplicate_rows_grouped=n_dup,
                       missing_pct=round(float(X.isna().mean().mean() * 100), 2), notes="; ".join(notes))
            loaded[c["name"]] = (c, Bundle(c["name"], X, y, classes, nominal, groups, notes))
        except Exception as e:
            rec.update(decision="exclude", reason=f"R5/R3: could not load or encode ({type(e).__name__}: {str(e)[:120]})")
        rows.append(rec)

    # R1, R2
    for rec in rows:
        if rec["decision"]: continue
        c, b = loaded[rec["dataset"]]
        counts = np.bincount(b.y, minlength=b.K)
        usable = int((b.X.isna().mean() <= CONFIG["COL_MISSING_DROP"]).sum())
        if counts.min() < CONFIG["MIN_CLASS_COUNT"]:
            rec.update(decision="exclude", reason=f"R1: smallest class has {counts.min()} samples")
        elif rec["n"] < CONFIG["MIN_SAMPLES"] or usable < CONFIG["MIN_FEATURES"]:
            rec.update(decision="exclude", reason=f"R2: n={rec['n']}, usable features={usable}")

    # R4a: Statlog and Cleveland are the same cohort if their rows overlap
    if "Statlog (Heart)" in loaded and "Heart Disease (Cleveland)" in loaded:
        ov = _statlog_overlap(loaded["Statlog (Heart)"][1].X, loaded["Heart Disease (Cleveland)"][1].X)
        for rec in rows:
            if rec["dataset"] == "Statlog (Heart)":
                rec["notes"] += f"; row overlap with Cleveland = {ov:.0%}"
                if ov > CONFIG["STATLOG_OVERLAP_MAX"]:
                    rec["cohort"] = "cleveland"
    # R4b: one dataset per cohort, keep the highest r4_priority (set in the registry before any result)
    prio = {c["name"]: c["r4_priority"] for c in CANDIDATES}
    for cohort in {r["cohort"] for r in rows}:
        members = [r for r in rows if r["cohort"] == cohort and not r["decision"]]
        if len(members) > 1:
            members.sort(key=lambda r: -prio[r["dataset"]])
            for r in members[1:]:
                r.update(decision="exclude", reason=f"R4: same cohort as '{members[0]['dataset']}'")

    for rec in rows:
        if not rec["decision"]:
            rec["decision"] = "include"
            bundles[rec["dataset"]] = loaded[rec["dataset"]][1]
    table = pd.DataFrame(rows)
    table.to_csv(os.path.join(CONFIG["SAVE_DIR"], "dataset_inclusion.csv"), index=False)
    if verbose:
        with pd.option_context("display.max_colwidth", 70, "display.width", 250):
            print(table[["dataset", "decision", "reason", "n", "d", "K", "IR", "duplicate_rows_grouped", "missing_pct"]].to_string(index=False))
        print(f"\n  {len(bundles)} datasets included -> {os.path.join(CONFIG['SAVE_DIR'], 'dataset_inclusion.csv')}")
    return bundles, table

## Cell 3b: Stage 2 datasets (replaces the v1 candidate list)

In [ ]:
# Cell 3b: Stage 2 (confirmation), new candidate datasets; replaces the v1 list
# None of these datasets was used in the v1 benchmark or in Stage 1. We fixed the corrections here before
# training, with the same principles as in v1: columns derived from the outcome or recorded after baseline are
# removed, other targets of the same record are removed, ambiguous labels are removed, and repeated
# measurements of a patient are grouped. Rules R1-R5 (Cell 3, audit_datasets) decide inclusion.
CANDIDATES.clear()


def _dedupe_columns(X):
    """ucimlrepo returns two columns with the same name for a few datasets (e.g. 'MDVP:Jitter(%)' and
    'MDVP:Jitter(Abs)' both as 'MDVP:Jitter'); make every name unique, order kept."""
    seen, cols = {}, []
    for c in X.columns:
        seen[c] = seen.get(c, 0) + 1
        cols.append(c if seen[c] == 1 else f"{c}_{seen[c]}")
    X = X.copy(); X.columns = cols
    return X


@candidate("Heart Failure Clinical Records", "hf_faisalabad", positive=["1"], nominal=[])
def _():
    X, T = _uci(519)
    return X.drop(columns=["time"]), _pick_target(T, "death_event"), None, \
        ["dropped 'time' (follow-up period: patients who die early have short follow-up -> outcome-derived)"]


@candidate("HCV Data", "hcv_hannover", positive=["disease"], nominal=["Sex"])
def _():
    from ucimlrepo import fetch_ucirepo
    d = fetch_ucirepo(id=571); X = d.data.features.copy(); t = d.data.targets["Category"].astype(str)
    keep = ~t.str.startswith("0s")  # '0s=suspect Blood Donor' is an ambiguous label
    y = t[keep].map(lambda v: "donor" if v.startswith("0=") else "disease")
    return X[keep].reset_index(drop=True), y.reset_index(drop=True), None, \
        [f"removed {int((~keep).sum())} 'suspect blood donor' rows (ambiguous label)",
         "label = liver disease (hepatitis, fibrosis or cirrhosis) vs blood donor"]


@candidate("Haberman's Survival", "haberman", positive=["2"])
def _():
    X, T = _uci(43); return X, _pick_target(T, "survival_status"), None, ["2 = died within 5 years of surgery"]


@candidate("Diabetic Retinopathy Debrecen", "dr_debrecen", positive=["1"])
def _():
    X, T = _uci(329); return _dedupe_columns(X), _pick_target(T, "Class"), None, ["1 = signs of diabetic retinopathy"]


@candidate("Thoracic Surgery", "thoracic_wroclaw", positive=["T"], nominal=["DGN", "PRE6", "PRE14"])
def _():
    X, T = _uci(277); return X, _pick_target(T, "Risk1Yr"), None, ["T = death within one year after surgery"]


@candidate("Breast Cancer (Ljubljana)", "breast_ljubljana", positive=["recurrence-events"],
           nominal=["age", "menopause", "tumor-size", "inv-nodes", "node-caps", "breast", "breast-quad", "irradiat"])
def _():
    X, T = _uci(14); return X, _pick_target(T, "Class"), None, ["positive = recurrence"]


@candidate("Fertility", "fertility", positive=["O"])
def _():
    X, T = _uci(244); return X, _pick_target(T, "diagnosis"), None, ["O = altered semen diagnosis"]


@candidate("Glioma Grading", "glioma_tcga", positive=["1"], nominal=["Race"])
def _():
    X, T = _uci(759); return X, _pick_target(T, "Grade"), None, ["1 = GBM (high grade), 0 = LGG"]


@candidate("Cervical Cancer Risk Factors", "cervical_caracas", positive=["1"])
def _():
    from ucimlrepo import fetch_ucirepo
    X = fetch_ucirepo(id=383).data.features.copy()
    y = X["Biopsy"]
    X = X.drop(columns=["Biopsy", "Hinselmann", "Schiller", "Citology"])
    return X, y, None, ["target = Biopsy; dropped Hinselmann, Schiller, Citology (other diagnostic tests of the same visit)"]


@candidate("Parkinson's (Little 2008)", "parkinsons_little", positive=["1"])
def _():
    from ucimlrepo import fetch_ucirepo
    d = fetch_ucirepo(id=174)
    X = _dedupe_columns(d.data.features)
    subj = d.data.ids["name"].astype(str).str.extract(r"(S\d+)")[0].to_numpy()  # phon_R01_S01_1 -> S01
    return X, _pick_target(d.data.targets, "status"), subj, \
        [f"grouped by subject ({len(set(subj))} subjects, several recordings each)"]


@candidate("Chronic Kidney Disease Risk (Bangladesh)", "ckd_bangladesh", positive=["ckd"],
           nominal=["bp (Diastolic)", "bp limit", "sg", "al", "rbc", "su", "pc", "pcc", "ba", "bgr", "bu", "sod", "sc", "pot",
                    "hemo", "pcv", "rbcc", "wbcc", "htn", "dm", "cad", "appet", "pe", "ane", "grf", "age"])
def _():
    X, T = _uci(857)
    return X.drop(columns=["stage", "affected"]), _pick_target(T, "class"), None, \
        ["dropped 'stage' (CKD stage, derived from the outcome) and 'affected'; features are recorded as ranges"]


@candidate("AIDS Clinical Trials (ACTG 175)", "actg175", positive=["1"], nominal=["trt", "race", "strat"])
def _():
    X, T = _uci(890)
    X = X.drop(columns=["time", "cd420", "cd820", "offtrt", "treat"])
    return X, _pick_target(T, "cid"), None, \
        ["dropped time (follow-up), cd420 / cd820 (CD4/CD8 at week 20, post-baseline), offtrt (post-baseline event), "
         "treat (duplicate of trt); baseline variables only"]


@candidate("Pediatric Appendicitis (Regensburg)", "appendicitis_regensburg", positive=["appendicitis"],
           nominal=["Sex", "Appendix_on_US", "Migratory_Pain", "Lower_Right_Abd_Pain", "Contralateral_Rebound_Tenderness",
                    "Coughing_Pain", "Nausea", "Loss_of_Appetite", "Neutrophilia", "Ketones_in_Urine", "RBC_in_Urine",
                    "WBC_in_Urine", "Dysuria", "Stool", "Peritonitis", "Psoas_Sign", "Ipsilateral_Rebound_Tenderness",
                    "US_Performed", "Free_Fluids", "Appendix_Wall_Layers", "Target_Sign", "Appendicolith", "Perfusion",
                    "Perforation", "Surrounding_Tissue_Reaction", "Appendicular_Abscess", "Abscess_Location",
                    "Pathological_Lymph_Nodes", "Lymph_Nodes_Location", "Bowel_Wall_Thickening",
                    "Conglomerate_of_Bowel_Loops", "Ileus", "Coprostasis", "Meteorism", "Enteritis", "Gynecological_Findings"])
def _():
    X, T = _uci(938)
    return X.drop(columns=["Length_of_Stay"]), _pick_target(T, "Diagnosis"), None, \
        ["target = Diagnosis; dropped Length_of_Stay (after the outcome) and the other targets Management, Severity"]


## Cell 4: splits, preprocessing, metrics

In [ ]:
# Cell 4: splits, preprocessing fitted per fold, metrics
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
                             confusion_matrix, f1_score, matthews_corrcoef, precision_score,
                             recall_score, roc_auc_score)
from sklearn.model_selection import (StratifiedGroupKFold, StratifiedKFold, StratifiedShuffleSplit)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, label_binarize


def make_folds(y, groups, n_splits, seed):
    """Stratified folds; grouped (patient-level) when groups are given."""
    if groups is not None:
        return list(StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=seed)
                    .split(np.zeros(len(y)), y, groups))
    return list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
                .split(np.zeros(len(y)), y))


def internal_split(y, groups, frac, seed):
    """Fit/validation split carved out of training data only."""
    if groups is not None:
        n = max(2, int(round(1 / frac)))
        return next(StratifiedGroupKFold(n_splits=n, shuffle=True, random_state=seed)
                    .split(np.zeros(len(y)), y, groups))
    return next(StratifiedShuffleSplit(n_splits=1, test_size=frac, random_state=seed)
                .split(np.zeros(len(y)), y))


class Preprocessor:
    """Fitted on the training part only: drop columns > 40 % missing, impute (median / mode),
    one-hot encode nominal and text columns, standardise everything."""

    def __init__(self, nominal):
        self.nominal = list(nominal)

    def fit(self, X: pd.DataFrame):
        miss = X.isna().mean()
        self.keep = [c for c in X.columns if miss[c] <= CONFIG["COL_MISSING_DROP"]]
        Xk = X[self.keep]
        self.cat = [c for c in self.keep if c in self.nominal or not pd.api.types.is_numeric_dtype(Xk[c])]
        self.num = [c for c in self.keep if c not in self.cat]
        try:
            ohe = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
        except TypeError:  # scikit-learn < 1.2
            ohe = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse=False)
        steps = []
        if self.num:
            steps.append(("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                                           ("sc", StandardScaler())]), self.num))
        if self.cat:
            steps.append(("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                           ("ohe", ohe), ("sc", StandardScaler())]), self.cat))
        self.ct = ColumnTransformer(steps)
        self.ct.fit(self._cast(Xk))
        return self

    def _cast(self, X):
        X = X.copy()
        for c in self.cat:
            X[c] = X[c].astype(object).where(X[c].notna(), np.nan).map(lambda v: v if pd.isna(v) else str(v))
        return X

    def transform(self, X):
        return np.asarray(self.ct.transform(self._cast(X[self.keep])), dtype=np.float32)


def fold_arrays(bundle, tr_idx, te_idx):
    pre = Preprocessor(bundle.nominal).fit(bundle.X.iloc[tr_idx])
    return (pre.transform(bundle.X.iloc[tr_idx]), bundle.y[tr_idx],
            pre.transform(bundle.X.iloc[te_idx]), bundle.y[te_idx])


def compute_metrics(y_true, proba, K):
    """Threshold-free and threshold-based metrics. Binary: class 1 = clinical condition."""
    proba = np.clip(np.asarray(proba, dtype=float), 1e-12, 1.0)
    proba = proba / proba.sum(1, keepdims=True)
    pred = proba.argmax(1)
    m = dict(accuracy=accuracy_score(y_true, pred),
             balanced_accuracy=balanced_accuracy_score(y_true, pred),
             mcc=matthews_corrcoef(y_true, pred))
    onehot = np.eye(K)[y_true]
    m["brier"] = float(np.mean(np.sum((proba - onehot) ** 2, axis=1)))  # multiclass Brier score (0 to 2)
    if K == 2:
        tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
        m.update(sensitivity=tp / max(tp + fn, 1), specificity=tn / max(tn + fp, 1),
                 precision=precision_score(y_true, pred, zero_division=0),
                 f1=f1_score(y_true, pred, zero_division=0))
        try: m["auroc"] = roc_auc_score(y_true, proba[:, 1])
        except ValueError: m["auroc"] = np.nan
        try: m["auprc"] = average_precision_score(y_true, proba[:, 1])
        except ValueError: m["auprc"] = np.nan
    else:
        cm = confusion_matrix(y_true, pred, labels=list(range(K)))
        spec = [(cm.sum() - cm[k].sum() - cm[:, k].sum() + cm[k, k]) / max(cm.sum() - cm[k].sum(), 1) for k in range(K)]
        m.update(sensitivity=recall_score(y_true, pred, average="macro", zero_division=0),
                 specificity=float(np.mean(spec)),
                 precision=precision_score(y_true, pred, average="macro", zero_division=0),
                 f1=f1_score(y_true, pred, average="macro", zero_division=0))
        present = np.unique(y_true)
        try: m["auroc"] = roc_auc_score(y_true, proba[:, present] / proba[:, present].sum(1, keepdims=True),
                                        multi_class="ovr", average="macro", labels=present) if len(present) > 2 else np.nan
        except ValueError: m["auroc"] = np.nan
        try: m["auprc"] = average_precision_score(label_binarize(y_true, classes=list(range(K)))[:, present],
                                                  proba[:, present], average="macro")
        except ValueError: m["auprc"] = np.nan
    return {k: float(v) for k, v in m.items()}


def selection_score(y_true, pred):
    return matthews_corrcoef(y_true, pred)  # CONFIG["SELECTION_METRIC"] = "mcc"


def sample_candidates(grid, n, seed):
    import itertools
    keys = list(grid.keys())
    combos = [dict(zip(keys, v)) for v in itertools.product(*[grid[k] for k in keys])]
    if n >= len(combos): return combos
    idx = np.random.default_rng(seed).choice(len(combos), size=n, replace=False)
    return [combos[i] for i in sorted(idx)]


def balanced_sample_weight(y):
    counts = np.bincount(y)
    w = len(y) / (len(counts) * np.maximum(counts, 1))
    return w[y]

## Cell 5: tuned single KAN, EGVW-KAN, ablation ladder

In [ ]:
# Cell 5: tuned single KAN, EGVW-KAN and the ablation ladder
# No leakage: every data-dependent choice (hyperparameters, imbalance tier, class weights,
# fitting guards, member weights) uses the outer training fold only. The outer test fold
# is used once, in predict, after all fitting is done.
# Ablation ladder. All rows use the same tuned hyperparameters in an outer fold:
#   A  KAN (tuned)            single KAN, unweighted loss                (= member 0 of B)
#   B  Ens-uniform            5 seeds, unweighted, mean of the logits
#   C  Ens+CW                 B + class weights by imbalance tier        (= B for balanced data)
#   D  Ens+CW+Boot            C, with the correction done by a balanced bootstrap (moderate/extreme); only the
#                             remaining exponent is used as class weights                (= C otherwise)
#   E  Ens+CW+Boot+Guards     D + fitting guards (internal split), uniform weights, refit
#   F  EGVW loss-only         members of E, weights from the validation loss
#   G  EGVW dispersion-only   members of E, weights from the logit dispersion
#   H  EGVW-KAN (proposed)    members of E, weights from loss + dispersion
#   I  EGVW-KAN fixed-CW      H with inverse-frequency weights on every dataset, no bootstrap
#   J  EGVW-KAN no-refit      H weights on the members trained on the fit split
#   K  EGVW-KAN double-CW     H as first defined: full class weights on top of the balanced bootstrap,
#                             so the correction is applied twice (= H without a bootstrap).
# Post-hoc sensitivity (no retraining): gamma grid, ensemble size M, standardised score terms.
import functools, time
import numpy as np


def _torch_single_thread():
    import torch
    torch.set_num_threads(1)
    try:
        import sys, tqdm as _t, kan  # noqa: F401
        _mk = sys.modules["kan.MultKAN"]  # the package re-exports a class with the same name
        _mk.tqdm = functools.partial(_t.tqdm, disable=True)  # no pykan progress bars
    except Exception:
        pass


def build_kan(d, K, p, seed):
    from kan import KAN
    kw = dict(width=[d, int(p["n_hidden"]), K], grid=int(p["grid"]), k=int(p["k"]), seed=int(seed), device="cpu")
    try:
        return KAN(**kw, auto_save=False, symbolic_enabled=False)
    except TypeError:  # older pykan without these arguments
        return KAN(**kw)


def train_kan(X, y, p, seed, class_w=None):
    """Train one KAN with L-bfgs. Returns None if training diverges (counted and reported)."""
    import torch
    try:
        model = build_kan(X.shape[1], p["_K"], p, seed)
        # torch.tensor copies into aligned memory; some BLAS kernels round differently depending on
        # the memory address, which would break run-to-run reproducibility
        Xt, yt = torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long)
        w = None if class_w is None else torch.tensor(class_w, dtype=torch.float32)
        loss_fn = torch.nn.CrossEntropyLoss(weight=w)
        # pykan only logs test_input/test_label; we pass the training rows so no other data is used
        model.fit({"train_input": Xt, "train_label": yt, "test_input": Xt, "test_label": yt},
                  opt="LBFGS", steps=int(p["steps"]), lr=float(p["lr"]), lamb=float(p["lamb"]),
                  loss_fn=loss_fn, log=int(p["steps"]) + 1)
        with torch.no_grad():
            if not torch.isfinite(model(Xt[:5])).all():
                return None
        return model
    except Exception:
        return None


def kan_logits(model, X, K):
    import torch
    if model is None:
        return np.zeros((len(X), K), dtype=np.float32)  # a failed member gives uninformative logits
    with torch.no_grad():
        z = model(torch.tensor(X, dtype=torch.float32)).numpy()
    return np.nan_to_num(z, nan=0.0, posinf=50.0, neginf=-50.0)


def ce(logits, y, class_w=None):
    z = logits - logits.max(1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(1, keepdims=True))
    nll = -logp[np.arange(len(y)), y]
    if class_w is None: return float(nll.mean())
    w = np.asarray(class_w)[y]
    return float((w * nll).sum() / w.sum())


def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)


# imbalance handling
def imbalance_tier(y, K):
    c = np.bincount(y, minlength=K).astype(float); c = c[c > 0]
    ir = float(c.max() / c.min()); t1, t2, t3 = CONFIG["IR_THRESHOLDS"]
    tier = "balanced" if ir < t1 else "mild" if ir < t2 else "moderate" if ir < t3 else "extreme"
    return tier, ir


def class_weights(y, K, mode, already_balanced=False):
    """mode: 'none' | 'tiered' (proposed) | 'balanced' (plain inverse frequency, ablation I).
    `y` = the original labels; the total correction of a tier is base ** exponent. already_balanced=True: the
    rows trained on are a balanced bootstrap of `y`, which already supplies exponent 1, so only the residual
    exponent is returned (moderate: none, extreme: 0.5), the correction is applied once, never twice."""
    if mode == "none": return None
    counts = np.bincount(y, minlength=K).astype(float); counts[counts == 0] = 1
    base = len(y) / (K * counts)
    expo = 1.0 if mode == "balanced" else CONFIG["TIER_EXPONENT"][imbalance_tier(y, K)[0]]
    if already_balanced: expo -= 1.0
    if expo <= 0: return None
    w = base ** expo; w = w / w.mean()
    return np.minimum(w, np.median(w) * CONFIG["WEIGHT_CAP"]).astype(np.float32)


def balanced_bootstrap(y, seed):
    rng = np.random.default_rng(seed)
    classes, counts = np.unique(y, return_counts=True)
    idx = np.concatenate([rng.choice(np.where(y == c)[0], size=counts.max(), replace=True) for c in classes])
    rng.shuffle(idx); return idx


def uses_bootstrap(y, K, mode):
    return mode == "tiered" and imbalance_tier(y, K)[0] in ("moderate", "extreme")


# member weights
def member_weights(val_loss, val_disp, scheme, gamma):
    L, D = np.asarray(val_loss, float), np.asarray(val_disp, float)
    if scheme == "uniform":
        return np.full(len(L), 1 / len(L))
    if scheme == "egvw_std":  # alternative score with standardised, unit-free terms
        z = lambda a: (a - a.mean()) / a.std() if a.std() > 0 else np.zeros_like(a)
        s = z(L) + z(D); w = np.exp(-gamma * (s - s.min()))
        return w / w.sum()
    s = {"loss": L, "disp": D, "egvw": L + D}[scheme]
    w = np.exp(-gamma * s / max(s.min(), 1e-8))
    return w / w.sum()


def ens_proba(logit_list, w):
    return softmax(np.tensordot(np.asarray(w), np.stack(logit_list), axes=1))


# hyperparameter search (single unweighted KAN, shared by all ladder rows)
def kan_search(bundle, tr_idx, seed):
    K = bundle.K
    folds = make_folds(bundle.y[tr_idx], None if bundle.groups is None else bundle.groups[tr_idx],
                       CONFIG["INNER_FOLDS"], seed + 1)
    arrays = [fold_arrays(bundle, tr_idx[a], tr_idx[b]) for a, b in folds]
    cands = sample_candidates(CONFIG["KAN_GRID"], CONFIG["N_CANDIDATES"], seed)
    table = []
    for p in cands:
        p = {**p, "_K": K}
        scores = []
        for i, (Xa, ya, Xb, yb) in enumerate(arrays):
            m = train_kan(Xa, ya, p, seed + 10 + i)
            scores.append(selection_score(yb, kan_logits(m, Xb, K).argmax(1)))
        table.append({**{k: v for k, v in p.items() if k != "_K"}, "inner_mcc": float(np.mean(scores))})
    best = max(range(len(table)), key=lambda i: table[i]["inner_mcc"])  # first best on ties
    return {**{k: v for k, v in table[best].items() if k != "inner_mcc"}, "_K": K}, table


# EGVW training on the internal split
def egvw_fit(Xtr, ytr, gtr, p, K, seed, cw_mode="tiered", boot_mode="tiered", sampler=None, double_cw=False):
    """sampler (Part 4 only): f(row indices into the training fold, member seed) -> (X, y) that replaces the
    balanced bootstrap, e.g. SMOTE-NC or random undersampling. None = the method as proposed.
    double_cw (ablation K only): the first definition, full class weights on top of the bootstrap."""
    fit_i, val_i = internal_split(ytr, gtr, CONFIG["INNER_VAL_FRAC"], seed + 7)
    Xf, yf, Xv, yv = Xtr[fit_i], ytr[fit_i], Xtr[val_i], ytr[val_i]
    boot = uses_bootstrap(yf, K, boot_mode)
    w = class_weights(yf, K, cw_mode)  # class weights applied to the validation rows
    w_tr = w if double_cw else class_weights(yf, K, cw_mode, already_balanced=boot and sampler is None)
    prior = np.bincount(yf, minlength=K) / len(yf)
    H0 = float(-(prior[prior > 0] * np.log(prior[prior > 0])).sum())
    out = dict(fit_models=[], configs=[], seeds=[], val_loss=[], val_disp=[], guards=[], failures=0)
    for m in range(CONFIG["N_MEMBERS"]):
        s = seed * 100 + m
        if sampler is None:
            idx = balanced_bootstrap(yf, s) if boot else np.arange(len(yf))
            Xm, ym = Xf[idx], yf[idx]
        else:
            Xm, ym = sampler(fit_i, s)
        def _run(pp):
            mod = train_kan(Xm, ym, pp, s, w_tr)
            return mod, ce(kan_logits(mod, Xm, K), ym, w_tr), ce(kan_logits(mod, Xv, K), yv, w)
        model, L_tr, L_va = _run(p); chosen, g = p, "none"
        if (L_va - L_tr) > CONFIG["OVERFIT_GAP"] and L_va > L_tr * CONFIG["OVERFIT_RATIO"]:
            alt = {**p, "lamb": p["lamb"] * CONFIG["OVERFIT_LAMB_MULT"]}
            m2, t2, v2 = _run(alt)
            if v2 < L_va: model, L_tr, L_va, chosen, g = m2, t2, v2, alt, "overfit"
        elif L_tr > CONFIG["UNDERFIT_MARGIN"] * H0:
            alt = {**p, "steps": min(int(p["steps"]) * CONFIG["UNDERFIT_STEPS_MULT"], CONFIG["UNDERFIT_STEPS_CAP"])}
            m2, t2, v2 = _run(alt)
            if t2 < L_tr: model, L_tr, L_va, chosen, g = m2, t2, v2, alt, "underfit"
        zv = kan_logits(model, Xv, K)
        out["fit_models"].append(model); out["configs"].append(chosen); out["seeds"].append(s)
        out["val_loss"].append(L_va); out["val_disp"].append(float(np.var(zv, axis=1).mean()))
        out["guards"].append(g); out["failures"] += model is None
    # refit on all training rows with each member's configuration; the weights do not change
    boot_full = uses_bootstrap(ytr, K, boot_mode)
    w_full = class_weights(ytr, K, cw_mode, already_balanced=boot_full and sampler is None and not double_cw)
    out["refit_models"] = []
    for cfg, s in zip(out["configs"], out["seeds"]):
        if sampler is None:
            idx = balanced_bootstrap(ytr, s) if boot_full else np.arange(len(ytr))
            Xr, yr = Xtr[idx], ytr[idx]
        else:
            Xr, yr = sampler(np.arange(len(ytr)), s)
        mod = train_kan(Xr, yr, cfg, s, w_full)
        out["refit_models"].append(mod); out["failures"] += mod is None
    out["tier"], out["ir"] = imbalance_tier(yf, K)
    out["boot_used"] = bool(sampler is None and (boot or boot_full))
    return out


def run_kan_family(bundle, tr_idx, te_idx, seed):
    """One outer fold -> probabilities for every ladder row + post-hoc variants."""
    _torch_single_thread()
    K = bundle.K
    Xtr, ytr, Xte, yte = fold_arrays(bundle, tr_idx, te_idx)
    gtr = None if bundle.groups is None else bundle.groups[tr_idx]
    probs, times, info = {}, {}, {}

    t0 = time.time(); p, search_table = kan_search(bundle, tr_idx, seed); t_search = time.time() - t0
    info["params"] = {k: v for k, v in p.items() if k != "_K"}
    tier, ir = imbalance_tier(ytr, K); info.update(tier=tier, ir=ir)
    seeds = [seed * 100 + m for m in range(CONFIG["N_MEMBERS"])]

    def ensemble(mode_cw, boot):
        w = class_weights(ytr, K, mode_cw, already_balanced=boot); t = time.time(); logits, fails = [], 0
        for s in seeds:
            idx = balanced_bootstrap(ytr, s) if boot else np.arange(len(ytr))
            mod = train_kan(Xtr[idx], ytr[idx], p, s, w); fails += mod is None
            logits.append(kan_logits(mod, Xte, K))
        return logits, time.time() - t, fails

    if CONFIG["RUN_ABLATION"]:
        LB, tB, fB = ensemble("none", False)
        probs["KAN (tuned)"] = softmax(LB[0]); times["KAN (tuned)"] = t_search + tB / len(seeds)
        probs["Ens-uniform"] = ens_proba(LB, np.full(len(LB), 1 / len(LB))); times["Ens-uniform"] = t_search + tB
        LC, tC, fC = (LB, tB, fB) if tier == "balanced" else ensemble("tiered", False)
        probs["Ens+CW"] = ens_proba(LC, np.full(len(LC), 1 / len(LC))); times["Ens+CW"] = t_search + tC
        LD, tD, fD = ensemble("tiered", True) if tier in ("moderate", "extreme") else (LC, tC, fC)
        probs["Ens+CW+Boot"] = ens_proba(LD, np.full(len(LD), 1 / len(LD))); times["Ens+CW+Boot"] = t_search + tD
        info["failures_BCD"] = int(fB + fC + fD)
    else:
        t = time.time(); m0 = train_kan(Xtr, ytr, p, seeds[0])
        probs["KAN (tuned)"] = softmax(kan_logits(m0, Xte, K)); times["KAN (tuned)"] = t_search + time.time() - t

    t = time.time(); E = egvw_fit(Xtr, ytr, gtr, p, K, seed, "tiered", "tiered"); tE = time.time() - t
    Lr = [kan_logits(m, Xte, K) for m in E["refit_models"]]
    Lf = [kan_logits(m, Xte, K) for m in E["fit_models"]]
    g = CONFIG["GAMMA"]
    wts = {sch: member_weights(E["val_loss"], E["val_disp"], sch, g) for sch in ("uniform", "loss", "disp", "egvw", "egvw_std")}
    for name, sch in [("Ens+CW+Boot+Guards", "uniform"), ("EGVW loss-only", "loss"),
                      ("EGVW dispersion-only", "disp"), ("EGVW-KAN", "egvw")]:
        probs[name] = ens_proba(Lr, wts[sch]); times[name] = t_search + tE
    probs["EGVW-KAN no-refit"] = ens_proba(Lf, wts["egvw"]); times["EGVW-KAN no-refit"] = t_search + tE
    info.update(egvw_weights=wts["egvw"].tolist(), val_loss=E["val_loss"], val_disp=E["val_disp"],
                guards=E["guards"], egvw_failures=E["failures"],
                weight_entropy=float(-(wts["egvw"] * np.log(wts["egvw"] + 1e-12)).sum() / np.log(len(wts["egvw"]))))

    # post-hoc sensitivity, no retraining
    sens = {}
    for gam in CONFIG["GAMMA_GRID"]:
        if gam != g:
            sens[f"EGVW-KAN [γ={gam}]"] = ens_proba(Lr, member_weights(E["val_loss"], E["val_disp"], "egvw", gam))
    for M in range(1, CONFIG["N_MEMBERS"]):
        sens[f"EGVW-KAN [M={M}]"] = ens_proba(Lr[:M], member_weights(E["val_loss"][:M], E["val_disp"][:M], "egvw", g))
    sens["EGVW-KAN [standardised score]"] = ens_proba(Lr, wts["egvw_std"])

    if CONFIG["RUN_ABLATION"]:
        t = time.time(); I = egvw_fit(Xtr, ytr, gtr, p, K, seed, "balanced", "never")
        LI = [kan_logits(m, Xte, K) for m in I["refit_models"]]
        probs["EGVW-KAN fixed-CW"] = ens_proba(LI, member_weights(I["val_loss"], I["val_disp"], "egvw", g))
        times["EGVW-KAN fixed-CW"] = t_search + time.time() - t
        info["failures_I"] = I["failures"]
        if E["boot_used"]:  # K: the first (double) correction; same as H when no bootstrap is used
            t = time.time(); Kd = egvw_fit(Xtr, ytr, gtr, p, K, seed, "tiered", "tiered", double_cw=True)
            LK = [kan_logits(m, Xte, K) for m in Kd["refit_models"]]
            probs["EGVW-KAN double-CW"] = ens_proba(LK, member_weights(Kd["val_loss"], Kd["val_disp"], "egvw", g))
            times["EGVW-KAN double-CW"] = t_search + time.time() - t
            info["failures_K"] = Kd["failures"]
        else:
            probs["EGVW-KAN double-CW"] = probs["EGVW-KAN"]; times["EGVW-KAN double-CW"] = times["EGVW-KAN"]
    # a KAN that fails to train gives neutral logits instead of stopping the unit; we count these failures
    info["train_failures_total"] = int(info.get("failures_BCD", 0) + E["failures"] + info.get("failures_I", 0)
                                       + info.get("failures_K", 0))

    return dict(yte=yte, probs=probs, sens=sens, times=times, info=info, search=search_table)

## Cell 6: classical baselines (same tuning budget)

In [ ]:
# Cell 6: classical baselines with the same tuning budget
# Same outer and inner folds, same N_CANDIDATES and selection metric (MCC), and
# class weights where the library supports them. The imbalance handling each
# method got is stored in the results (column "imbalance").
import inspect, time, warnings
import numpy as np

BASELINE_GRIDS = {
    "Logistic regression": {"C": [0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30, 100, 300, 1000]},
    "SVM (RBF)":           {"C": [0.1, 1, 10, 100], "gamma": ["scale", 0.001, 0.01, 0.1]},
    "KNN":                 {"n_neighbors": [3, 5, 7, 9, 15], "weights": ["uniform", "distance"], "p": [1, 2]},
    "Random forest":       {"n_estimators": [100, 300, 500], "max_depth": [None, 5, 10, 20], "min_samples_leaf": [1, 2, 4]},
    "XGBoost":             {"n_estimators": [100, 200, 300], "max_depth": [3, 6, 9], "learning_rate": [0.03, 0.1, 0.2], "subsample": [0.8, 1.0]},
    "LightGBM":            {"n_estimators": [100, 200, 300], "num_leaves": [7, 15, 31], "learning_rate": [0.03, 0.1, 0.2], "min_child_samples": [5, 10, 20]},
    "CatBoost":            {"depth": [4, 6, 8], "learning_rate": [0.03, 0.1], "iterations": [200, 500], "l2_leaf_reg": [1, 3, 9]},
    "MLP":                 {"hidden_layer_sizes": [(32,), (64, 32), (128, 64)], "alpha": [1e-4, 1e-3, 1e-2], "learning_rate_init": [1e-3, 1e-2]},
}


def make_baseline(name, params, seed, K):
    """Returns (estimator, how class imbalance is handled: 'class_weight' | 'sample_weight' | 'none')."""
    if name == "Logistic regression":
        from sklearn.linear_model import LogisticRegression
        return LogisticRegression(C=params["C"], class_weight="balanced", max_iter=5000), "class_weight"
    if name == "SVM (RBF)":
        from sklearn.svm import SVC
        return SVC(C=params["C"], gamma=params["gamma"], class_weight="balanced", probability=True, random_state=seed), "class_weight"
    if name == "KNN":
        from sklearn.neighbors import KNeighborsClassifier
        return KNeighborsClassifier(**params), "none"
    if name == "Random forest":
        from sklearn.ensemble import RandomForestClassifier
        return RandomForestClassifier(**params, class_weight="balanced", random_state=seed, n_jobs=1), "class_weight"
    if name == "XGBoost":
        from xgboost import XGBClassifier
        return XGBClassifier(**params, tree_method="hist", n_jobs=1, random_state=seed, verbosity=0,
                             eval_metric="mlogloss" if K > 2 else "logloss"), "sample_weight"
    if name == "LightGBM":
        from lightgbm import LGBMClassifier
        return LGBMClassifier(**params, class_weight="balanced", random_state=seed, n_jobs=1, verbose=-1), "class_weight"
    if name == "CatBoost":
        from catboost import CatBoostClassifier
        return CatBoostClassifier(**params, auto_class_weights="Balanced", random_seed=seed, verbose=0,
                                  thread_count=1, allow_writing_files=False), "class_weight"
    if name == "MLP":
        from sklearn.neural_network import MLPClassifier
        est = MLPClassifier(**params, max_iter=1000, early_stopping=True, random_state=seed)
        sw_ok = "sample_weight" in inspect.signature(est.fit).parameters  # scikit-learn >= 1.7
        return est, "sample_weight" if sw_ok else "none"
    raise KeyError(name)


def _fit_predict(name, params, seed, K, Xa, ya, Xb):
    est, how = make_baseline(name, params, seed, K)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        if how == "sample_weight":
            est.fit(Xa, ya, sample_weight=balanced_sample_weight(ya))
        else:
            est.fit(Xa, ya)
        proba = est.predict_proba(Xb)
    full = np.zeros((len(Xb), K)); full[:, est.classes_.astype(int)] = proba  # classes missing in a fold
    return full, how


def run_baselines(bundle, tr_idx, te_idx, seed):
    K = bundle.K
    Xtr, ytr, Xte, yte = fold_arrays(bundle, tr_idx, te_idx)
    folds = make_folds(bundle.y[tr_idx], None if bundle.groups is None else bundle.groups[tr_idx],
                       CONFIG["INNER_FOLDS"], seed + 1)
    inner = [fold_arrays(bundle, tr_idx[a], tr_idx[b]) for a, b in folds]
    probs, times, info = {}, {}, {}
    for name, grid in BASELINE_GRIDS.items():
        t = time.time()
        try:
            cands = sample_candidates(grid, CONFIG["N_CANDIDATES"], seed)
            scores = []
            for p in cands:
                s = []
                for Xa, ya, Xb, yb in inner:
                    try: s.append(selection_score(yb, _fit_predict(name, p, seed, K, Xa, ya, Xb)[0].argmax(1)))
                    except Exception: s.append(-1.0)
                scores.append(np.mean(s))
            best = cands[int(np.argmax(scores))]
            probs[name], how = _fit_predict(name, best, seed, K, Xtr, ytr, Xte)
            info[name] = dict(params=best, imbalance=how)
        except Exception as e:
            info[name] = dict(error=f"{type(e).__name__}: {str(e)[:150]}")
        times[name] = time.time() - t
    return dict(yte=yte, probs=probs, times=times, info=info)

## Cell 7: the eight KAN variants

In [ ]:
# Cell 7: the eight KAN variants
# We first used full-batch AdamW with ExponentialLR(0.8) per epoch for 35 epochs. The learning rate
# dropped below 1% of its start by epoch 20 and the variants were under-trained.
# Now: full-batch AdamW, cosine annealing over VARIANT_EPOCHS, early stopping on an internal
# validation split of the training fold (patience VARIANT_PATIENCE), best weights restored
# (input LayerNorm of five variants replaced by an identity, see below),
# class-weighted loss, and the same N_CANDIDATES, inner folds and MCC selection as the other methods.
import importlib.util, os, sys, time, types
import numpy as np

FCKAN_MODELS = os.path.join(os.path.abspath("./github_kan_repos/FC_KAN"), "models")


def _load_module(mod_name, file_path):
    if mod_name in sys.modules: return sys.modules[mod_name]
    spec = importlib.util.spec_from_file_location(mod_name, file_path)
    module = importlib.util.module_from_spec(spec); sys.modules[mod_name] = module
    spec.loader.exec_module(module); return module


def _fake_models_pkg(models_dir):
    # fc_kan.py and skan.py do `from models.functions import *`; we skip the repo's __init__.py
    if "models" in sys.modules and hasattr(sys.modules["models"], "_fckan_stub"): return
    pkg = types.ModuleType("models"); pkg.__path__ = [models_dir]; pkg._fckan_stub = True
    sys.modules["models"] = pkg
    f = os.path.join(models_dir, "functions.py")
    if os.path.exists(f): _load_module("models.functions", f)


def _build_variant(name, models_dir, layers, **hp):
    if name == "EfficientKAN":
        return _load_module("fckan.efficient_kan", os.path.join(models_dir, "efficient_kan.py")).EfficientKAN(layers, **hp)
    if name == "FastKAN":
        return _load_module("fckan.fast_kan", os.path.join(models_dir, "fast_kan.py")).FastKAN(layers, **hp)
    if name == "FasterKAN":
        return _load_module("fckan.faster_kan", os.path.join(models_dir, "faster_kan.py")).FasterKAN(layers, **hp)
    if name == "BSRBF-KAN":
        return _load_module("fckan.bsrbf_kan", os.path.join(models_dir, "bsrbf_kan.py")).BSRBF_KAN(layers, **hp)
    if name == "GottliebKAN":
        return _load_module("fckan.gottlieb_kan", os.path.join(models_dir, "gottlieb_kan.py")).GottliebKAN(layers, **hp)
    if name == "SKAN":
        _fake_models_pkg(models_dir)
        return _load_module("fckan.skan", os.path.join(models_dir, "skan.py")).SKAN(layers, device="cpu", **hp)
    if name == "WavKAN":
        return _load_module("fckan.wav_kan", os.path.join(models_dir, "wav_kan.py")).KAN(layers)
    if name == "FC-KAN":
        _fake_models_pkg(models_dir)
        # ['dog', 'rbf'] + 'quadratic' is the combination in the repository README; the
        # 'shifted_softplus' and 'sigmoid' branches of FC_KANLayer.forward use undefined names in the original code.
        return _load_module("fckan.fc_kan", os.path.join(models_dir, "fc_kan.py")).FC_KAN(
            layers, ["dog", "rbf"], combined_type="quadratic", **hp)
    raise KeyError(name)


# Change for tabular data, applied to every variant that has it: the FC-KAN code normalises each sample's
# input vector with LayerNorm, which is fine for images. On tabular data it removes each patient's overall
# level and spread, so a patient high on every feature looks the same as one low on every feature
# (Pima, SKAN: MCC 0.26 with it, 0.49 without). Features are already standardised per fold, so we replace
# the input LayerNorm by an identity. LayerNorm on hidden layers (GottliebKAN) stays.
INPUT_LAYERNORM_VARIANTS = {"FastKAN", "FasterKAN", "BSRBF-KAN", "SKAN", "FC-KAN"}


def _drop_input_layernorm(name, model, d_in):
    import torch
    if name not in INPUT_LAYERNORM_VARIANTS: return
    ln = getattr(model.layers[0], "layernorm", None)
    if not isinstance(ln, torch.nn.LayerNorm) or tuple(ln.normalized_shape) != (d_in,):
        raise RuntimeError(f"{name}: expected an input LayerNorm over {d_in} features - repository changed?")
    model.layers[0].layernorm = _Contiguous()


def _Contiguous():
    import torch
    class Contiguous(torch.nn.Module):  # identity; some variants call .view() on the input
        def forward(self, x): return x.contiguous()
    return Contiguous()


_TRAIN_GRID = {"lr": [1e-3, 3e-3, 1e-2], "weight_decay": [1e-4, 1e-3], "hidden": [16, 32]}
VARIANT_GRIDS = {
    "EfficientKAN": {"grid_size": [3, 5], "spline_order": [3]},
    "FastKAN":      {"num_grids": [4, 8]},
    "FasterKAN":    {"num_grids": [4, 8]},
    "BSRBF-KAN":    {"grid_size": [3, 5], "spline_order": [3]},
    "GottliebKAN":  {"spline_order": [2, 3]},
    "SKAN":         {"basis_function": ["shifted_softplus", "arctan"]},
    "WavKAN":       {},
    "FC-KAN":       {"grid_size": [3, 5]},
}


def train_variant(name, hp, X, y, groups, K, seed):
    import torch
    torch.manual_seed(seed); np.random.seed(seed)
    arch = {k: v for k, v in hp.items() if k not in _TRAIN_GRID}
    fi, vi = internal_split(y, groups, 0.2, seed + 3)
    Xf, yf = torch.tensor(X[fi], dtype=torch.float32), torch.tensor(y[fi], dtype=torch.long)
    Xv, yv = torch.tensor(X[vi], dtype=torch.float32), torch.tensor(y[vi], dtype=torch.long)
    model = _build_variant(name, FCKAN_MODELS, [X.shape[1], int(hp["hidden"]), K], **arch)
    _drop_input_layernorm(name, model, X.shape[1])
    w = torch.tensor(class_weights(y[fi], K, "balanced"), dtype=torch.float32)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=hp["lr"], weight_decay=hp["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CONFIG["VARIANT_EPOCHS"])
    best, best_state, wait = np.inf, None, 0
    for _ in range(CONFIG["VARIANT_EPOCHS"]):
        model.train(); opt.zero_grad()
        loss = loss_fn(model(Xf), yf)
        if not torch.isfinite(loss): break
        loss.backward(); opt.step(); sched.step()
        model.eval()
        with torch.no_grad(): v = float(loss_fn(model(Xv), yv))
        if v < best - 1e-5:
            best, wait = v, 0
            best_state = {k: t.detach().clone() for k, t in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CONFIG["VARIANT_PATIENCE"]: break
    if best_state is not None: model.load_state_dict(best_state)
    model.eval(); return model


def variant_proba(model, X):
    import torch
    with torch.no_grad():
        return softmax(np.nan_to_num(model(torch.tensor(X, dtype=torch.float32)).numpy()))


def run_variants(bundle, tr_idx, te_idx, seed):
    _torch_single_thread()
    K = bundle.K
    Xtr, ytr, Xte, yte = fold_arrays(bundle, tr_idx, te_idx)
    gtr = None if bundle.groups is None else bundle.groups[tr_idx]
    folds = make_folds(bundle.y[tr_idx], gtr, CONFIG["INNER_FOLDS"], seed + 1)
    inner = [(fold_arrays(bundle, tr_idx[a], tr_idx[b]), None if gtr is None else gtr[a]) for a, b in folds]
    probs, times, info = {}, {}, {}
    if not os.path.isdir(FCKAN_MODELS):
        raise FileNotFoundError(f"FC_KAN repository not found at {FCKAN_MODELS} - run Cell 1 (Internet ON)")
    for name, arch_grid in VARIANT_GRIDS.items():
        t = time.time()
        try:
            cands = sample_candidates({**arch_grid, **_TRAIN_GRID}, CONFIG["N_CANDIDATES"], seed)
            scores = []
            for hp in cands:
                s = []
                for (Xa, ya, Xb, yb), ga in inner:
                    try: s.append(selection_score(yb, variant_proba(train_variant(name, hp, Xa, ya, ga, K, seed), Xb).argmax(1)))
                    except Exception: s.append(-1.0)
                scores.append(np.mean(s))
            best = cands[int(np.argmax(scores))]
            probs[name] = variant_proba(train_variant(name, best, Xtr, ytr, gtr, K, seed), Xte)
            info[name] = dict(params=best)
        except Exception as e:
            info[name] = dict(error=f"{type(e).__name__}: {str(e)[:150]}")
        times[name] = time.time() - t
    return dict(yte=yte, probs=probs, times=times, info=info)

## Cell 7b: Part 4, resampling instead of class weights

In [ ]:
# Cell 7b: Part 4, resampling instead of the class weights
# Same folds, tuned KAN configuration (from Part 1), internal split, member seeds,
# guards and EGVW weights as EGVW-KAN. Only the imbalance correction changes:
#   EGVW-KAN + SMOTE-NC            synthetic minority rows (SMOTE-NC; SMOTE or SMOTEN with one feature type)
#   EGVW-KAN + RUS                 random undersampling of each class to the smallest class
#   EGVW-KAN (no correction)       no class weights and no resampling
#   EGVW-KAN (no correction) + prior threshold
#                                  same model, decision = argmax p_k / pi_k (pi = class shares in the training
#                                  fold), i.e. threshold moving. Its Brier score is not meaningful.
# Resampling changes only the rows a member trains on (not the internal validation rows, not the test fold),
# and each member draws its own sample, like the balanced bootstrap.
import numpy as np
import pandas as pd

RESAMPLING_METHODS = ["EGVW-KAN + SMOTE-NC", "EGVW-KAN + RUS", "EGVW-KAN (no correction)",
                      "EGVW-KAN (no correction) + prior threshold"]


def _smote_rows(Xraw, y, pre, seed, k_max=5):
    """Synthetic rows for every non-majority class, generated in the raw feature space of the training rows
    (numeric columns median-imputed and standardised for the distance, nominal columns mode-imputed) and then
    passed through the preprocessor already fitted on the training fold, so test rows are transformed exactly
    as in Parts 1–3. Returns (synthetic feature matrix, synthetic labels, note)."""
    from imblearn.over_sampling import SMOTE, SMOTENC, SMOTEN, RandomOverSampler
    import warnings; warnings.filterwarnings("ignore", category=FutureWarning)  # pandas fillna downcasting warning
    A = Xraw[pre.keep].reset_index(drop=True).copy()
    num, cat = list(pre.num), list(pre.cat)
    for c in num:
        A[c] = pd.to_numeric(A[c], errors="coerce"); A[c] = A[c].fillna(A[c].median()).fillna(0.0)
    for c in cat:
        v = A[c].astype(object).where(A[c].notna(), np.nan)
        A[c] = v.fillna(v.mode().iloc[0] if v.notna().any() else "missing").map(str)
    mu = A[num].mean() if num else None
    sd = A[num].std(ddof=0).replace(0, 1).fillna(1) if num else None
    Z = A.copy()
    if num: Z[num] = (A[num] - mu) / sd
    counts = np.bincount(y); present = counts[counts > 0]
    k = int(min(k_max, present.min() - 1))
    if k < 1:  # a class with one row cannot be interpolated
        smp, note = RandomOverSampler(random_state=seed), "random oversampling (a class had 1 row)"
    elif not cat:
        smp, note = SMOTE(k_neighbors=k, random_state=seed), f"SMOTE k={k}"
    elif not num:
        smp, note = SMOTEN(k_neighbors=k, random_state=seed), f"SMOTEN k={k}"
    else:
        smp, note = SMOTENC(categorical_features=[Z.columns.get_loc(c) for c in cat], k_neighbors=k,
                            random_state=seed), f"SMOTE-NC k={k}"
    Zr, yr = smp.fit_resample(Z, y)
    Zr = pd.DataFrame(Zr, columns=Z.columns)
    new = Zr.iloc[len(Z):].copy()  # imbalanced-learn appends the synthetic rows after the original ones
    if new.empty:
        return np.zeros((0, 0), np.float32), np.zeros(0, int), note
    if num:
        new[num] = new[num].astype(float) * sd + mu
    return pre.transform(new), np.asarray(yr[len(Z):], dtype=int), note


def run_resampling(bundle, tr_idx, te_idx, seed, params=None):
    """One outer fold -> probabilities for the four rows above. `params` = the configuration Part 1 selected on
    this fold (read from its output); without it (self-test only) the inner search is run here."""
    _torch_single_thread()
    K = bundle.K
    pre = Preprocessor(bundle.nominal).fit(bundle.X.iloc[tr_idx])  # same as fold_arrays()
    Xtr, Xte = pre.transform(bundle.X.iloc[tr_idx]), pre.transform(bundle.X.iloc[te_idx])
    ytr, yte = bundle.y[tr_idx], bundle.y[te_idx]
    Xraw = bundle.X.iloc[tr_idx].reset_index(drop=True)
    gtr = None if bundle.groups is None else bundle.groups[tr_idx]
    info, times, probs, notes = {}, {}, {}, set()
    if params is None:
        p, _ = kan_search(bundle, tr_idx, seed); info["params_source"] = "searched here (no Part 1 output)"
    else:
        p = {**params, "_K": K}; info["params_source"] = "Part 1"
    info["params"] = {k: v for k, v in p.items() if k != "_K"}

    def smote_sampler(idx, s):
        Xs, ys, note = _smote_rows(Xraw.iloc[idx], ytr[idx], pre, s); notes.add(note)
        if len(ys) == 0: return Xtr[idx], ytr[idx]
        return np.vstack([Xtr[idx], Xs]).astype(np.float32), np.concatenate([ytr[idx], ys])

    def rus_sampler(idx, s):
        rng = np.random.default_rng(s); y = ytr[idx]
        cls, cnt = np.unique(y, return_counts=True)
        keep = np.sort(np.concatenate([rng.choice(np.where(y == c)[0], cnt.min(), replace=False) for c in cls]))
        return Xtr[idx][keep], y[keep]

    g = CONFIG["GAMMA"]

    def fit_rows(name, sampler):
        t = time.time()
        E = egvw_fit(Xtr, ytr, gtr, p, K, seed, "none", "never", sampler=sampler)
        L = [kan_logits(m, Xte, K) for m in E["refit_models"]]
        w = member_weights(E["val_loss"], E["val_disp"], "egvw", g)
        info[name] = dict(weights=w.tolist(), guards=E["guards"], failures=int(E["failures"]))
        return ens_proba(L, w), time.time() - t, int(E["failures"])

    fails = 0
    probs["EGVW-KAN + SMOTE-NC"], times["EGVW-KAN + SMOTE-NC"], f1 = fit_rows("smote", smote_sampler)
    probs["EGVW-KAN + RUS"], times["EGVW-KAN + RUS"], f2 = fit_rows("rus", rus_sampler)
    P0, t0, f3 = fit_rows("none", None)
    probs["EGVW-KAN (no correction)"], times["EGVW-KAN (no correction)"] = P0, t0
    prior = np.bincount(ytr, minlength=K) / len(ytr)
    Q = P0 / np.maximum(prior, 1e-12); Q = Q / Q.sum(1, keepdims=True)
    probs["EGVW-KAN (no correction) + prior threshold"] = Q
    times["EGVW-KAN (no correction) + prior threshold"] = t0
    info["smote_variant"] = sorted(notes)
    info["train_failures_total"] = f1 + f2 + f3
    return dict(yte=yte, probs=probs, sens={}, times=times, info=info, search=None)

## Cell 7c: Part 5, EGVWC-KAN v2

In [ ]:
# Cell 7c: Part 5, EGVWC-KAN v2 (development stage, see preregistration_v2.md)
# What went wrong in v1 (Parts 1 and 4): the overfitting guard (lambda x 5) makes members flat on small datasets.
# A flat member has almost no logit dispersion, so the v1 score (validation loss + dispersion) gives it the
# largest weight, and that score uses a 20% internal split (about 20 rows on the smallest datasets). v2 changes:
#   members       5 KANs, each trained on a stratified bootstrap of the outer training fold (group bootstrap
#                 when the dataset has patient or duplicate groups) with the full tiered class weights.
#                 No balanced bootstrap (it leaves almost no out-of-bag minority rows), no guards, no refit.
#   score         balanced log-loss of each member on its own out-of-bag (OOB) rows (mean of per-class NLL).
#   gate          weight 0 for a member that predicts one class on its OOB rows or failed to train.
#   shrinkage     gamma_eff = gamma * n_oob / (n_oob + C_SHRINK): small OOB sets give nearly uniform weights.
#   weights       w_m ~ exp(-gamma_eff * (s_m / min s - 1)) over members that pass the gate; uniform if none does.
# Same folds, tuned KAN configuration (from Part 1) and member seeds as v1. The outer test fold is used once,
# for prediction. OOB rows are rows of the training fold.
# Rows (all built from the same five members, so they differ only in the weighting):
#   v2                     gate + shrinkage + OOB balanced loss
#   v2 - gate              no gate
#   v2 - shrink            gamma_eff = gamma
#   v2 uniform             bagged ensemble, uniform weights
#   v2 uniform + gate      uniform weights over members that pass the gate
#   v2 [v1 score]          v1 score (class-weighted loss + logit dispersion) on the OOB rows, no gate or shrinkage
# Post-hoc sensitivity (no retraining): gamma in V2_GAMMA_GRID x C_SHRINK in V2_SHRINK_GRID, with the gate.
import time
import numpy as np

V2_METHODS = ["v2", "v2 − gate", "v2 − shrink", "v2 uniform", "v2 uniform + gate", "v2 [v1 score]"]


def v2_bootstrap(y, groups, seed):
    """Stratified bootstrap of the training rows (each class resampled to its own size); with groups, whole
    groups are resampled so that rows of one patient are never split between in-bag and out-of-bag."""
    rng = np.random.default_rng(seed)
    if groups is None:
        idx = np.concatenate([rng.choice(np.where(y == c)[0], size=(y == c).sum(), replace=True) for c in np.unique(y)])
    else:
        ug = np.unique(groups); pick = rng.choice(ug, size=len(ug), replace=True)
        pos = {g: np.where(groups == g)[0] for g in ug}
        idx = np.concatenate([pos[g] for g in pick])
    oob = np.setdiff1d(np.arange(len(y)), idx)
    return idx, oob


def balanced_nll(logits, y, K):
    """Mean over the classes present of the mean negative log-likelihood within the class."""
    z = logits - logits.max(1, keepdims=True)
    logp = z - np.log(np.exp(z).sum(1, keepdims=True))
    nll = -logp[np.arange(len(y)), y]
    return float(np.mean([nll[y == c].mean() for c in np.unique(y)]))


def v2_weights(score, gated, n_oob, gamma, c_shrink, use_gate=True, shrink=True):
    s = np.asarray(score, float); keep = ~np.asarray(gated) if use_gate else np.ones(len(s), bool)
    if not keep.any(): return np.full(len(s), 1 / len(s))
    g = gamma * (n_oob / (n_oob + c_shrink) if shrink else 1.0)
    w = np.zeros(len(s)); ref = max(s[keep].min(), 1e-8)
    w[keep] = np.exp(-g * (s[keep] / ref - 1.0))
    return w / w.sum()


def run_v2(bundle, tr_idx, te_idx, seed, params=None):
    """One outer fold -> probabilities for the v2 rows. `params` = the configuration Part 1 selected on this fold."""
    _torch_single_thread()
    K = bundle.K
    Xtr, ytr, Xte, yte = fold_arrays(bundle, tr_idx, te_idx)
    gtr = None if bundle.groups is None else bundle.groups[tr_idx]
    info, times, probs = {}, {}, {}
    t_search = 0.0
    if params is None:
        t0 = time.time(); p, _ = kan_search(bundle, tr_idx, seed); t_search = time.time() - t0
        info["params_source"] = "searched here (no Part 1 output)"
    else:
        p = {**params, "_K": K}; info["params_source"] = "Part 1"
    info["params"] = {k: v for k, v in p.items() if k != "_K"}
    cw = class_weights(ytr, K, "tiered")  # full tiered exponent (no balanced bootstrap)
    t0 = time.time()
    Lte, bal, v1s, gated, n_oob, fails = [], [], [], [], [], 0
    for m in range(CONFIG["N_MEMBERS"]):
        s = seed * 100 + m
        idx, oob = v2_bootstrap(ytr, gtr, s)
        mod = train_kan(Xtr[idx], ytr[idx], p, s, cw); fails += mod is None
        zo = kan_logits(mod, Xtr[oob], K); yo = ytr[oob]
        Lte.append(kan_logits(mod, Xte, K))
        bal.append(balanced_nll(zo, yo, K) if len(yo) else np.inf)
        v1s.append((ce(zo, yo, cw) + float(np.var(zo, axis=1).mean())) if len(yo) else np.inf)
        one_class = len(np.unique(yo)) > 1 and len(np.unique(zo.argmax(1))) == 1
        gated.append(bool(mod is None or one_class or not len(yo))); n_oob.append(len(yo))
    t_fit = time.time() - t0 + t_search
    n = float(np.median(n_oob)); G, C = CONFIG["GAMMA"], CONFIG["V2_C_SHRINK"]
    W = {"v2": v2_weights(bal, gated, n, G, C),
         "v2 − gate": v2_weights(bal, gated, n, G, C, use_gate=False),
         "v2 − shrink": v2_weights(bal, gated, n, G, C, shrink=False),
         "v2 uniform": np.full(len(bal), 1 / len(bal)),
         "v2 uniform + gate": v2_weights(np.ones(len(bal)), gated, n, 0.0, C),
         "v2 [v1 score]": member_weights(v1s, np.zeros(len(v1s)), "loss", G)}
    for name, w in W.items():
        probs[name] = ens_proba(Lte, w); times[name] = t_fit
    sens = {}
    for gam in CONFIG["V2_GAMMA_GRID"]:
        for c in CONFIG["V2_SHRINK_GRID"]:
            if (gam, c) != (G, C):
                sens[f"v2 [γ={gam}, C={c}]"] = ens_proba(Lte, v2_weights(bal, gated, n, gam, c))
    info.update(weights={k: v.tolist() for k, v in W.items()}, oob_balanced_nll=bal, v1_score=v1s,
                gated=gated, n_oob=n_oob, n_gated=int(sum(gated)), train_failures_total=int(fails))
    return dict(yte=yte, probs=probs, sens=sens, times=times, info=info, search=None)


def analyse_v2():
    """Stage 4 for Part 5: v2 rows vs EGVW-KAN (H) and Ens+CW (C) of Part 1 on identical folds (complete datasets)."""
    import pandas as pd
    from scipy import stats
    R = pd.read_json(F_RES, lines=True)
    ref = ["EGVW-KAN", "Ens+CW", "Ens-uniform", "KAN (tuned)"]
    S = R[R.method.isin(V2_METHODS + ref)]
    n = S.groupby(["dataset", "method"]).size().unstack()
    full = n.index[(n.reindex(columns=V2_METHODS + ref) == CONFIG["REPEATS"] * CONFIG["OUTER_FOLDS"]).all(1)]
    M = S[S.dataset.isin(full)].groupby(["dataset", "method"]).mcc.mean().unstack()
    print(f"  {len(full)} datasets complete for every row")
    if len(full) < 2: return M
    rows = []
    for m in V2_METHODS:
        for r in ("EGVW-KAN", "Ens+CW"):
            d = M[m] - M[r]
            p = stats.wilcoxon(M[m], M[r], zero_method="zsplit").pvalue if (d != 0).any() else 1.0
            rows.append(dict(row=m, vs=r, mcc=M[m].mean(), ref_mcc=M[r].mean(), delta=d.mean(),
                             wins=int((d > 1e-9).sum()), losses=int((d < -1e-9).sum()), p=p))
    T = pd.DataFrame(rows).round(4)
    T.to_csv(os.path.join(CONFIG["SAVE_DIR"], "v2_vs_v1.csv"), index=False)
    M.round(4).to_csv(os.path.join(CONFIG["SAVE_DIR"], "v2_mcc_by_dataset.csv"))
    print(T.to_string(index=False))
    return T


## Cell 7d: Stage 2, v2-final inside every KAN unit

In [ ]:
# Cell 7d: Stage 2, v2-final rows inside every KAN unit
# v2-final is the v2 definition frozen in preregistration_v2.md (Stage 1 kept the defaults: gamma = 1, C = 100, gate on).
# It uses the KAN configuration chosen by the inner search of this unit (out["info"]["params"]), i.e. the
# same configuration as EGVW-KAN v1 and the ablation rows on the same outer fold.
if "_run_kan_family_v1" not in globals():
    _run_kan_family_v1 = run_kan_family


def run_kan_family(bundle, tr_idx, te_idx, seed):
    out = _run_kan_family_v1(bundle, tr_idx, te_idx, seed)
    v = run_v2(bundle, tr_idx, te_idx, seed, params=out["info"]["params"])
    for name, P in v["probs"].items():
        out["probs"][name] = P; out["times"][name] = out["times"].get("KAN (tuned)", 0.0) + v["times"][name]
    out["sens"].update(v["sens"])
    out["info"]["v2"] = v["info"]
    out["info"]["train_failures_total"] = int(out["info"].get("train_failures_total", 0) + v["info"]["train_failures_total"])
    return out


## Cell 8: runner (repeated nested CV, checkpoint and resume, leakage self-test)

In [ ]:
# Cell 8: runner (repeated nested CV, checkpoint and resume, parallel workers)
# One unit = (dataset, repeat, outer fold, group), group in {kan, baselines, variants}.
# A finished unit is appended to results.jsonl, info.jsonl and done.txt, and its test-fold
# probabilities go to preds/*.npz. An interrupted session loses only the units still running.
# To resume in a new Kaggle session: Save Version, then in the next session Add Input >
# Your Work > this notebook's output. Files under RESUME_GLOB are merged first.
import glob, json, os, re, shutil, time
import numpy as np
import pandas as pd

OUT = CONFIG["SAVE_DIR"]
PRED_DIR = os.path.join(OUT, "preds"); os.makedirs(PRED_DIR, exist_ok=True)
F_RES, F_INFO, F_DONE = (os.path.join(OUT, f) for f in ("results.jsonl", "info.jsonl", "done.txt"))
MAIN_KAN = {"EGVW-KAN", "KAN (tuned)"}


def _slug(s): return re.sub(r"[^A-Za-z0-9]+", "_", s).strip("_")


def matching_dirs(verbose=False):
    """Attached result folders (earlier sessions, other notebooks) whose protocol.json equals ours."""
    mine = json.load(open(os.path.join(OUT, "protocol.json")))
    keep = []
    for d in sorted(set(glob.glob(CONFIG["RESUME_GLOB"], recursive=True))):
        if not os.path.isdir(d) or os.path.abspath(d) == os.path.abspath(OUT): continue
        pf = os.path.join(d, "protocol.json")
        if os.path.exists(pf) and json.load(open(pf)) == mine:
            keep.append(d)
        elif verbose:
            print(f"  SKIPPED {d}: protocol differs from this notebook (smoke run or other settings)")
    return keep


def _stale_units(d):
    """EGVW-KAN / Part 4 units of another METHOD_VERSION and KAN-variant units of another VARIANT_VERSION:
    never merged. Baseline units do not depend on either and are always kept."""
    stale, f = set(), os.path.join(d, "results.jsonl")
    if os.path.exists(f):
        for line in open(f):
            try: r = json.loads(line)
            except Exception: continue
            grp = r["key"].rsplit("|", 1)[1]
            if (grp in ("kan", "resampling") and r.get("method_version") != CONFIG["METHOD_VERSION"]) or \
               (grp == "variants" and r.get("variant_version") != CONFIG["VARIANT_VERSION"]) or \
               (grp == "v2" and r.get("v2_version") != CONFIG.get("V2_VERSION")):
                stale.add(r["key"])
    return stale


def merge_previous_runs(verbose=True):
    """Merge finished units from attached inputs: earlier sessions of this notebook and the
    outputs of the other notebooks. A folder is used only if its protocol.json matches ours."""
    for d in matching_dirs(verbose):
        if not os.path.exists(os.path.join(d, "done.txt")): continue
        done_here = set(open(F_DONE).read().split()) if os.path.exists(F_DONE) else set()
        prev_done = set(open(os.path.join(d, "done.txt")).read().split()) if os.path.exists(os.path.join(d, "done.txt")) else set()
        stale = _stale_units(d)
        new = prev_done - done_here - stale
        if stale and verbose:
            print(f"  {len(stale)} units in {d} use an older method / variant definition - not merged")
        if not new: continue
        for fname in ("results.jsonl", "info.jsonl"):
            src = os.path.join(d, fname)
            if not os.path.exists(src): continue
            with open(src) as fi, open(os.path.join(OUT, fname), "a") as fo:
                for line in fi:
                    if json.loads(line)["key"] in new: fo.write(line)
        for k in new:  # prediction files of the merged units only
            f = os.path.join(d, "preds", k.replace("|", "__") + ".npz")
            if os.path.exists(f) and not os.path.exists(os.path.join(PRED_DIR, os.path.basename(f))): shutil.copy(f, PRED_DIR)
        with open(F_DONE, "a") as fo: fo.write("\n".join(sorted(new)) + "\n")
        print(f"  merged {len(new)} finished units from {d}")


def part1_params():
    """{'<dataset>|r<r>|f<f>|kan': params} for every fold Part 1 has finished (merged into info.jsonl)."""
    out = {}
    done = set(open(F_DONE).read().split()) if os.path.exists(F_DONE) else set()
    if not os.path.exists(F_INFO): return out
    for line in open(F_INFO):
        try: r = json.loads(line)
        except Exception: continue
        inf = r.get("info") or {}
        if r.get("group") == "kan" and r.get("key") in done and isinstance(inf, dict) and inf.get("params"):
            out[r["key"]] = inf["params"]
    return out


def in_shard(name):
    """True if this notebook's shard runs `name` (shard in Cell 2). Smoke runs ignore the split."""
    if CONFIG["SMOKE"] or not CONFIG.get("SHARD"): return True
    return (name in CONFIG["SHARD_A_DATASETS"]) == (CONFIG["SHARD"] == "A")


def build_tasks(bundles):
    unknown = set(CONFIG.get("SHARD_A_DATASETS", [])) - {c["name"] for c in CANDIDATES}
    if unknown:
        raise ValueError(f"SHARD_A_DATASETS has names that are not candidates (typo?): {sorted(unknown)}")
    groups = [g for g, on in (("kan", CONFIG["RUN_KAN"]), ("baselines", CONFIG["RUN_BASELINES"]),
                               ("variants", CONFIG["RUN_VARIANTS"]),
                               ("resampling", CONFIG.get("RUN_RESAMPLING", False)), ("v2", CONFIG.get("RUN_V2", False))) if on]
    tasks = []
    for r in range(CONFIG["REPEATS"]):
        for name, b in bundles.items():
            if not in_shard(name): continue  # folds depend only on the dataset and the seed
            folds = make_folds(b.y, b.groups, CONFIG["OUTER_FOLDS"], CONFIG["BASE_SEED"] + 1000 * r)
            for f, (tr, te) in enumerate(folds):
                for g in groups:
                    tasks.append(dict(key=f"{_slug(name)}|r{r}|f{f}|{g}", dataset=name, repeat=r, fold=f,
                                      group=g, tr=tr, te=te, seed=CONFIG["BASE_SEED"] + 1000 * r + 10 * f))
    return tasks


def run_unit(bundle, t):
    fn = {"kan": run_kan_family, "baselines": run_baselines, "variants": run_variants,
          "resampling": run_resampling, "v2": run_v2}[t["group"]]
    t0 = time.time()
    try:
        extra = {"params": t["params"]} if t.get("params") is not None else {}
        out = fn(bundle, t["tr"], t["te"], t["seed"], **extra)
        out["error"] = None
    except Exception as e:
        import traceback
        out = dict(yte=bundle.y[t["te"]], probs={}, times={}, info={}, error=traceback.format_exc()[-1500:])
    out["wall"] = time.time() - t0
    return t, out


def _family(group, method):
    if group == "resampling": return "resampling"
    if group == "v2": return "v2"
    if group == "baselines": return "baseline"
    if group == "variants": return "kan_variant"
    return "main" if method in MAIN_KAN else "ablation"


def unit_problems(out):
    """Why a unit cannot count as finished (empty list = complete)."""
    probs = []
    if out.get("error"): probs.append(out["error"].strip().splitlines()[-1])
    if not out.get("probs"): probs.append("no predictions")
    for name, inf in (out.get("info") or {}).items():
        if isinstance(inf, dict) and inf.get("error"): probs.append(f"{name}: {inf['error']}")
    return probs


def store(bundle, t, out):
    """Record a unit. Only a complete unit is written to results.jsonl and done.txt; an incomplete one
    is logged to info.jsonl and reruns next session, so a transient failure never silently drops a model."""
    rows, K = [], bundle.K
    import hashlib
    problems = unit_problems(out)
    if problems:
        with open(F_INFO, "a") as fo:
            fo.write(json.dumps({"key": t["key"], "dataset": t["dataset"], "repeat": t["repeat"], "fold": t["fold"],
                                 "group": t["group"], "info": {}, "error": " | ".join(problems)[:2000],
                                 "incomplete": True, "wall": out.get("wall")}, default=str) + "\n")
        return 0
    base = dict(key=t["key"], dataset=t["dataset"], repeat=t["repeat"], fold=t["fold"],
                method_version=CONFIG["METHOD_VERSION"], variant_version=CONFIG["VARIANT_VERSION"],
                v2_version=CONFIG.get("V2_VERSION"),
                n_train=len(t["tr"]), n_test=len(t["te"]),
                # the same in every notebook when the folds are the same (checked in the analysis)
                fold_hash=hashlib.md5(np.sort(np.asarray(t["te"])).tobytes()).hexdigest()[:12])
    for method, P in out["probs"].items():
        rows.append({**base, "method": method, "family": _family(t["group"], method),
                     "time_sec": out["times"].get(method, np.nan), **compute_metrics(out["yte"], P, K)})
    for method, P in out.get("sens", {}).items():
        rows.append({**base, "method": method, "family": "sensitivity", "time_sec": np.nan,
                     **compute_metrics(out["yte"], P, K)})
    with open(F_RES, "a") as fo:
        for r in rows: fo.write(json.dumps(r, default=float) + "\n")
    with open(F_INFO, "a") as fo:
        fo.write(json.dumps({**base, "group": t["group"], "info": out.get("info", {}), "error": out["error"],
                             "search": out.get("search"), "wall": out["wall"]}, default=str) + "\n")
    allp = {**out["probs"], **out.get("sens", {})}
    np.savez_compressed(os.path.join(PRED_DIR, t["key"].replace("|", "__") + ".npz"),
                        y=out["yte"], te_idx=t["te"], methods=np.array(list(allp.keys())),
                        probs=np.stack(list(allp.values())) if allp else np.zeros((0,)))
    with open(F_DONE, "a") as fo: fo.write(t["key"] + "\n")
    return len(rows)


# The self-test checks code paths, not accuracy, so it uses a small search budget (every method and
# ladder row still runs). The benchmark itself uses the full budget.
SELFTEST_BUDGET = dict(N_CANDIDATES=3, INNER_FOLDS=2, VARIANT_EPOCHS=60, VARIANT_PATIENCE=10)


def _all_probs(out):
    return {**out["probs"], **out.get("sens", {})}


def _failures(out):
    inf = out.get("info") or {}
    return int(inf.get("train_failures_total", 0)) if isinstance(inf, dict) else 0


def selftest_dataset(bundles):
    """Smallest dataset on which every component is active (class weights and balanced bootstrap switch on
    only in the moderate / extreme imbalance tiers); the smallest dataset overall if there is none."""
    active = [k for k, b in bundles.items() if imbalance_tier(b.y, b.K)[0] in ("moderate", "extreme")]
    return min(active or list(bundles), key=lambda k: len(bundles[k].y))


def leakage_self_test(bundle, groups=("kan", "baselines", "variants", "resampling", "v2")):
    """check a reviewer can rerun, on one outer fold. Two tests; both must pass.

    1. sentinel (decisive on any machine): every test-fold label is replaced by the impossible class K.
       Any training, weighting, guard or selection step that reads a test label then either crashes
       (cross-entropy / indexing with class K) or produces K+1 output columns. pass = the run completes
       with the same number of training failures as a normal run and every output has exactly K columns.
    2. permutation with a rerun control: the fold is trained normally, then with the test labels permuted.
       Predictions must be identical. If they differ, the normal run is repeated: if the repeat also
       differs, the machine does not reproduce training bit-for-bit (a numerical property of the hardware /
       BLAS library, not leakage), which is reported; if the repeat is identical, the change was caused by
       the test labels -> leak.
    Returns dict(ok=bool, summary=str)."""
    import dataclasses
    saved = {k: CONFIG[k] for k in SELFTEST_BUDGET}
    CONFIG.update(SELFTEST_BUDGET)
    try:
        tr, te = make_folds(bundle.y, bundle.groups, CONFIG["OUTER_FOLDS"], CONFIG["BASE_SEED"])[0]
        K, seed = bundle.K, CONFIG["BASE_SEED"]
        y_perm = bundle.y.copy(); y_perm[te] = np.random.default_rng(0).permutation(y_perm[te])
        y_sent = bundle.y.copy(); y_sent[te] = K
        permuted, sentinel = dataclasses.replace(bundle, y=y_perm), dataclasses.replace(bundle, y=y_sent)
        fns = {"kan": run_kan_family, "baselines": run_baselines, "variants": run_variants,
               "resampling": run_resampling, "v2": run_v2}
        groups = [g for g in groups if CONFIG.get({"kan": "RUN_KAN", "baselines": "RUN_BASELINES",
                                                   "variants": "RUN_VARIANTS", "resampling": "RUN_RESAMPLING", "v2": "RUN_V2"}[g], False)]
        ok, notes = True, []
        for g in groups:
            t0 = time.time()
            a = fns[g](bundle, tr, te, seed)
            if unit_problems(a):  # fail if nothing was checked
                ok = False; print(f"  {g}: could not run - {unit_problems(a)[0]}"); continue
            pa = _all_probs(a)

            # 1. sentinel test
            try:
                s = fns[g](sentinel, tr, te, seed); s_err = unit_problems(s)
            except Exception as e:
                s, s_err = None, [f"{type(e).__name__}: {e}"]
            bad = list(s_err)
            if s is not None and not s_err:
                ps = _all_probs(s)
                bad += [f"{m}: missing" for m in pa if m not in ps]
                bad += [f"{m}: {np.shape(ps[m])[1]} output columns (expected {K})" for m in ps
                        if np.ndim(ps[m]) != 2 or np.shape(ps[m])[1] != K]
                bad += [f"{m}: non-finite output" for m in ps if not np.isfinite(ps[m]).all()]
                if _failures(s) != _failures(a):
                    bad.append(f"{_failures(s) - _failures(a):+d} training failures when test labels were impossible")
            if bad:
                ok = False
                for b in bad[:10]: print(f"  LEAK (sentinel) {g}: {b}")
            print(f"  {g}: sentinel test {'PASSED' if not bad else 'FAILED'} ({len(pa)} methods)")

            # 2. permutation test with a rerun control
            b = fns[g](permuted, tr, te, seed); p = _all_probs(b)
            d_perm = {m: float(np.abs(pa[m] - p[m]).max()) if m in p else np.inf for m in pa}
            changed = [m for m in pa if d_perm[m] > 1e-6]
            if not changed:
                print(f"  {g}: permutation test PASSED - predictions bit-identical after permuting test labels")
                notes.append(f"{g}: sentinel PASSED; permutation PASSED (bit-identical)")
            else:
                a2_out = fns[g](bundle, tr, te, seed); a2 = _all_probs(a2_out)
                d_rerun = {m: float(np.abs(pa[m] - a2[m]).max()) for m in pa}
                leak = [m for m in changed if d_rerun[m] <= 1e-6]  # reproducible, but changed with the labels
                noisy = [m for m in changed if d_rerun[m] > 1e-6]
                for m in leak:
                    print(f"  LEAK (permutation) {g}/{m}: identical reruns agree, but permuting test labels "
                          f"changed predictions by {d_perm[m]:.2e}")
                if leak: ok = False
                if noisy:
                    print(f"  {g}: NOT BIT-REPRODUCIBLE on this machine - {len(noisy)} of {len(pa)} methods give "
                          f"different predictions when the SAME fold is trained twice with the SAME seed")
                    print(f"      largest change, identical rerun: {max(d_rerun[m] for m in noisy):.2e};  "
                          f"after permuting test labels: {max(d_perm[m] for m in noisy):.2e}")
                    # a large change usually means a discrete choice flipped (inner-CV winner, guard branch)
                    sel = [(lab, (o.get("info") or {}).get("params"), (o.get("info") or {}).get("guards"))
                           for lab, o in (("run 1", a), ("permuted", b), ("run 2", a2_out))]
                    if any(x[1] is not None for x in sel):
                        for lab, prm, grd in sel: print(f"      {lab:<9} selected {prm}  guards {grd}")
                    print(f"      The permutation test cannot separate leakage from numerical noise for these methods; "
                          f"the sentinel test above is the decisive check.")
                notes.append(f"{g}: sentinel {'PASSED' if not bad else 'FAILED'}; permutation "
                             + ("FAILED" if leak else "not bit-reproducible on this machine "
                                f"(rerun change up to {max(d_rerun.values()):.1e})"))
            print(f"  {g}: done in {time.time() - t0:.0f} s")
        summary = "; ".join(notes)
        print("  LEAKAGE SELF-TEST:", "PASSED - no method uses test-fold labels" if ok else "FAILED")
        return dict(ok=ok, summary=summary)
    finally:
        CONFIG.update(saved)


def run_all(bundles):
    """Keeps N_JOBS units in flight. A unit is started only if its expected duration (the longest
    time seen for the same dataset and group) fits in the remaining budget; at HARD_STOP_HOURS the
    workers are killed so the Kaggle session always ends, and saves its output, before 12 h."""
    from concurrent.futures import FIRST_COMPLETED, wait
    from joblib.externals.loky import get_reusable_executor
    merge_previous_runs()
    done = set(open(F_DONE).read().split()) if os.path.exists(F_DONE) else set()
    queue = [t for t in build_tasks(bundles) if t["key"] not in done]
    mine = sorted(n for n in bundles if in_shard(n))
    print(f"  shard {CONFIG.get('SHARD')}: {len(mine)} of {len(bundles)} datasets - {mine}")
    waiting = []
    if any(t["group"] in ("resampling", "v2") for t in queue):
        # Part 4 uses the KAN configuration Part 1 selected on the same fold, so the hyperparameters are the
        # same and any difference comes from the imbalance handling. Folds that Part 1 has not finished wait.
        P1 = part1_params()
        for t in queue:
            if t["group"] in ("resampling", "v2"): t["params"] = P1.get(t["key"].rsplit("|", 1)[0] + "|kan")
        waiting = [t for t in queue if t["group"] in ("resampling", "v2") and t["params"] is None]
        queue = [t for t in queue if not (t["group"] in ("resampling", "v2") and t["params"] is None)]
        print(f"  Part 1 configurations found for {len(P1)} folds; {len(waiting)} Part 4 units wait for Part 1 "
              f"(attach Part 1's latest output to run them)")
    total, n_todo = len(queue) + len(done), len(queue)
    print(f"  {len(done)} units already finished, {n_todo} to run, {CONFIG['N_JOBS']} parallel workers")
    t0 = CONFIG.get("SESSION_START", time.time())
    budget, hard = CONFIG["MAX_RUNTIME_HOURS"] * 3600, CONFIG["HARD_STOP_HOURS"] * 3600
    ex = get_reusable_executor(max_workers=CONFIG["N_JOBS"], timeout=None)
    inflight, seen, finished, last_print, stop_submit, failed, retries = {}, {}, 0, 0, False, [], {}
    while queue or inflight:
        now = time.time() - t0
        while queue and len(inflight) < CONFIG["N_JOBS"] and not stop_submit:
            t = queue[0]
            # a unit retried after a worker crash runs alone: a unit killed together with the one that crashed
            # then succeeds, and a memory-hungry unit gets all of the RAM
            if any(x.get("isolate") for x in inflight.values()) or (t.get("isolate") and inflight):
                break
            guess = CONFIG["UNSEEN_UNIT_GUESS_HOURS"] * 3600  # (dataset, group) without a timing yet
            est = seen.get((t["dataset"], t["group"]), guess if now > budget - guess else 0)
            if now + est > budget:
                stop_submit = True; break
            inflight[ex.submit(run_unit, bundles[t["dataset"]], t)] = queue.pop(0)
        if not inflight: break
        ready, _ = wait(list(inflight), timeout=60, return_when=FIRST_COMPLETED)
        broken = False
        for fut in ready:
            t = inflight.pop(fut)
            try:
                t, out = fut.result()
            except Exception as e:  # the worker process died (e.g. out of memory)
                broken = broken or "Broken" in type(e).__name__ or "Terminated" in type(e).__name__
                retries[t["key"]] = retries.get(t["key"], 0) + 1
                if retries[t["key"]] <= 1:
                    queue.append({**t, "isolate": True})  # one retry in this session, alone, at the end
                    print(f"  [WORKER CRASH] {t['key']}: {type(e).__name__} - retrying once, alone, later this session")
                else:
                    failed.append(t["key"])
                    print(f"  [WORKER CRASH AGAIN] {t['key']}: skipped this session (likely out of memory)")
                continue
            k = (t["dataset"], t["group"]); seen[k] = max(seen.get(k, 0), out["wall"])
            problems = unit_problems(out)
            if problems:
                failed.append(t["key"]); store(bundles[t["dataset"]], t, out)
                print(f"  [INCOMPLETE - will rerun next session] {t['key']}: {problems[0][:200]}")
            else:
                store(bundles[t["dataset"]], t, out); finished += 1
                nf = (out.get("info") or {}).get("train_failures_total", 0)
                if nf:
                    print(f"  [WARNING] {t['key']}: {nf} KAN trainings diverged and were replaced by neutral "
                          f"predictions (look for 'lstsq failed' above). If this recurs for a dataset, report it.")
        if broken:  # a dead worker breaks the pool: requeue the running units and restart
            for fut, t in list(inflight.items()):
                queue.append(t); inflight.pop(fut)
            ex = get_reusable_executor(max_workers=CONFIG["N_JOBS"], timeout=None, kill_workers=True)
            print("  worker pool restarted")
        now = time.time() - t0
        if ready and (now - last_print > 300 or not (queue or inflight)):
            last_print = now
            print(f"  {len(done) + finished}/{total} units | session {now/3600:.2f} h | in flight {len(inflight)}")
        if now > hard:
            print("  hard stop reached - killing units in flight (they rerun next session).")
            ex.shutdown(wait=False, kill_workers=True); break
    left = n_todo - finished
    if failed:
        print(f"  {len(failed)} units were INCOMPLETE this session (errors above; details in info.jsonl). "
              f"If the same units fail again next session, the error is not transient - fix it before continuing.")
    if left == 0 and waiting:
        print(f"  NOT finished: {len(waiting)} Part 4 units still wait for Part 1. Save Version; when Part 1 has run "
              f"further, attach its latest output and run this notebook again.")
    else:
        print("  finished." if left == 0 else
              f"  {left} units left - Save Version, then resume in a new session (see the header cell).")

## Cell 9: analysis (tables, statistics, figures)

In [ ]:
# Cell 9: analysis (tables, statistics, figures)
# Works on partial results and can be re-run at any time. Output goes to SAVE_DIR/tables and SAVE_DIR/figures.
import glob, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import confusion_matrix

TAB = os.path.join(CONFIG["SAVE_DIR"], "tables"); FIG = os.path.join(CONFIG["SAVE_DIR"], "figures")
os.makedirs(TAB, exist_ok=True); os.makedirs(FIG, exist_ok=True)
CONTROL = "EGVW-KAN"
LADDER = ["KAN (tuned)", "Ens-uniform", "Ens+CW", "Ens+CW+Boot", "Ens+CW+Boot+Guards",
          "EGVW loss-only", "EGVW dispersion-only", "EGVW-KAN", "EGVW-KAN fixed-CW", "EGVW-KAN no-refit",
          "EGVW-KAN double-CW"]
METRICS = ["mcc", "balanced_accuracy", "auroc", "auprc", "brier", "sensitivity", "specificity", "f1", "accuracy"]


def load_results():
    """(R, I), or (None, None) when no unit has finished yet (e.g. Part 4 waiting for Part 1)."""
    f_res, f_info = (os.path.join(CONFIG["SAVE_DIR"], f) for f in ("results.jsonl", "info.jsonl"))
    if not os.path.exists(f_res) or os.path.getsize(f_res) == 0:
        return None, None
    R = pd.read_json(f_res, lines=True)
    I = pd.read_json(f_info, lines=True) if os.path.exists(f_info) and os.path.getsize(f_info) else pd.DataFrame(
        columns=["key", "dataset", "group", "info", "error"])
    return R, I


def complete_only(R, methods):
    """Keep datasets on which every listed method has all R×K folds (paired comparisons)."""
    need = CONFIG["REPEATS"] * CONFIG["OUTER_FOLDS"]
    sub = R[R.method.isin(methods)]
    cnt = sub.groupby(["dataset", "method"]).size().unstack()
    ok = cnt.index[(cnt.reindex(columns=methods) == need).all(axis=1)]
    dropped = sorted(set(sub.dataset) - set(ok))
    if dropped: print(f"  (incomplete, left out of paired statistics for now: {dropped})")
    return sub[sub.dataset.isin(ok)]


def dataset_means(R, metric):
    return R.groupby(["dataset", "method"])[metric].mean().unstack()


def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); adj = np.empty_like(p); run = 0
    for i, j in enumerate(o):
        run = max(run, min(1.0, (len(p) - i) * p[j])); adj[j] = run
    return adj


def nemenyi_cd(k, N, alpha=0.05):
    q = stats.studentized_range.ppf(1 - alpha, k, np.inf) / np.sqrt(2)
    return q * np.sqrt(k * (k + 1) / (6 * N))


def corrected_ttest(d, n_train, n_test):
    """Nadeau & Bengio (2003) corrected repeated k-fold t-test on fold-level differences."""
    d = np.asarray(d, float); J = len(d)
    v = d.var(ddof=1)
    if v == 0: return 1.0 if d.mean() == 0 else 0.0
    t = d.mean() / np.sqrt((1 / J + n_test / n_train) * v)
    return float(2 * stats.t.sf(abs(t), J - 1))


def compare_to_control(M, control=CONTROL, higher_better=True):
    """Across-dataset: Friedman, mean ranks, Holm-Wilcoxon vs control, Nemenyi CD, Bayesian signed-rank."""
    M = M.dropna()
    ranks = M.rank(axis=1, ascending=not higher_better).mean().sort_values()
    fr = stats.friedmanchisquare(*[M[c] for c in M.columns]) if M.shape[1] > 2 else None
    rows = []
    for c in M.columns:
        if c == control: continue
        d = M[control] - M[c]
        try: p = stats.wilcoxon(M[control], M[c], zero_method="zsplit").pvalue
        except ValueError: p = 1.0
        bay = (np.nan, np.nan, np.nan)
        try:
            import baycomp
            bay = baycomp.two_on_multiple(M[control].values, M[c].values, rope=0.01)
        except Exception:
            pass
        rows.append(dict(method=c, mean_diff=d.mean(), median_diff=d.median(),
                         wins=int((d > 1e-9).sum() if higher_better else (d < -1e-9).sum()),
                         losses=int((d < -1e-9).sum() if higher_better else (d > 1e-9).sum()),
                         p_wilcoxon=p, p_bayes_control_better=bay[0] if higher_better else bay[2],
                         p_bayes_rope=bay[1]))
    T = pd.DataFrame(rows); T["p_holm"] = holm(T.p_wilcoxon); T = T.sort_values("p_wilcoxon")
    cd = nemenyi_cd(M.shape[1], M.shape[0])
    return ranks, fr, T, cd


def cd_diagram(ranks, cd, title, path):
    """Demšar (2006) critical-difference diagram: best methods on the right, bars join
    groups of methods whose mean ranks differ by less than the Nemenyi CD."""
    ranks = ranks.sort_values(); names, r = list(ranks.index), ranks.values; k = len(r)
    half = (k + 1) // 2
    cliques = []  # largest groups whose rank range is below CD
    for i in range(k):
        j = i
        while j + 1 < k and r[j + 1] - r[i] < cd: j += 1
        if j > i and not any(a <= i and j <= b for a, b in cliques): cliques.append((i, j))
    y_ax = 0.7
    y_lab = y_ax + 0.25 + 0.07 * len(cliques)
    fig_h = y_lab + 0.26 * half + 0.25
    fig, ax = plt.subplots(figsize=(9, fig_h))
    ax.set_xlim(k + 0.2 + 3.2, 1 - 0.2 - 3.2); ax.set_ylim(fig_h, 0); ax.axis("off")
    ax.hlines(y_ax, 1, k, color="k", lw=1)
    for t in range(1, k + 1):
        ax.vlines(t, y_ax - 0.06, y_ax, color="k", lw=1); ax.text(t, y_ax - 0.12, str(t), ha="center", fontsize=7)
    ax.hlines(0.25, 1, 1 + cd, color="k", lw=1.5); ax.text(1 + cd / 2, 0.18, f"CD = {cd:.2f}", ha="center", fontsize=7)
    for i in range(k):  # better half on the right, the rest on the left
        right = i < half
        j = i if right else k - 1 - i
        y = y_lab + 0.26 * j
        xe = 1 - 0.2 if right else k + 0.2
        ax.plot([r[i], r[i], xe], [y_ax, y, y], color="0.3", lw=0.8)
        ax.text(xe - 0.1 if right else xe + 0.1, y, f"{names[i]} ({r[i]:.2f})", va="center",
                ha="left" if right else "right", fontsize=7.5, fontweight="bold" if names[i] == CONTROL else "normal")
    for n, (a, b) in enumerate(cliques):
        yy = y_ax + 0.14 + 0.07 * n
        ax.hlines(yy, r[a] - 0.03, r[b] + 0.03, color="k", lw=2.5)
    ax.set_title(title, fontsize=9); fig.tight_layout(); fig.savefig(path, dpi=200); plt.close(fig)


def main_comparison(R, metric="mcc"):
    methods = sorted(set(R[R.family.isin(["main", "baseline", "kan_variant"])].method))
    sub = complete_only(R, methods)
    if sub.empty: print("  no dataset complete yet"); return
    M = dataset_means(sub, metric)
    hb = metric != "brier"
    if CONTROL not in M.columns:  # e.g. the baselines notebook analysed alone
        print(f"\n== {metric.upper()} - {M.shape[0]} datasets, {CONTROL} not attached yet: mean per method ==")
        print(M.mean().sort_values(ascending=not hb).round(4).to_string())
        return
    ranks, fr, T, cd = compare_to_control(M, higher_better=hb)
    # per-dataset table with the corrected resampled t-test against the control (dagger = p < 0.05)
    tab = M.copy().round(3).astype(object)
    for ds in M.index:
        c = sub[(sub.dataset == ds) & (sub.method == CONTROL)].sort_values(["repeat", "fold"])
        ntr, nte = c.n_train.mean(), c.n_test.mean()
        for m in M.columns:
            if m == CONTROL: continue
            o = sub[(sub.dataset == ds) & (sub.method == m)].sort_values(["repeat", "fold"])
            if corrected_ttest(c[metric].values - o[metric].values, ntr, nte) < 0.05:
                tab.loc[ds, m] = f"{M.loc[ds, m]:.3f}†"
    tab.loc["Mean"] = M.mean().round(3); tab.loc["Mean rank"] = M.rank(axis=1, ascending=not hb).mean().round(2)
    tab = tab[ranks.index]
    tab.to_csv(os.path.join(TAB, f"table_main_{metric}.csv"))
    T.round(4).to_csv(os.path.join(TAB, f"tests_vs_control_{metric}.csv"), index=False)
    try: tab.to_latex(os.path.join(TAB, f"table_main_{metric}.tex"))
    except Exception: pass
    cd_diagram(ranks, cd, f"Mean rank on {metric} over {M.shape[0]} datasets", os.path.join(FIG, f"cd_{metric}.png"))
    print(f"\n== {metric.upper()} - {M.shape[0]} datasets × {M.shape[1]} methods ==")
    if fr: print(f"  Friedman χ²({M.shape[1]-1}) = {fr.statistic:.2f}, p = {fr.pvalue:.2g};  Nemenyi CD = {cd:.2f}")
    print("  mean ranks:", ", ".join(f"{m} {r:.2f}" for m, r in ranks.items()))
    print(T[["method", "mean_diff", "wins", "losses", "p_wilcoxon", "p_holm", "p_bayes_control_better", "p_bayes_rope"]].round(4).to_string(index=False))
    return tab, T


def ablation_table(R, metric="mcc"):
    rows_present = [m for m in LADDER if m in set(R.method)]
    sub = complete_only(R, rows_present)
    if sub.empty: return
    M = dataset_means(sub, metric)[rows_present]
    # reference row of each comparison: A, B, C, D, E form a chain; F, G and H weight E's members
    # differently (each vs E); I and J change H (each vs H).
    ref = {"Ens-uniform": "KAN (tuned)", "Ens+CW": "Ens-uniform", "Ens+CW+Boot": "Ens+CW",
           "Ens+CW+Boot+Guards": "Ens+CW+Boot", "EGVW loss-only": "Ens+CW+Boot+Guards",
           "EGVW dispersion-only": "Ens+CW+Boot+Guards", "EGVW-KAN": "Ens+CW+Boot+Guards",
           "EGVW-KAN fixed-CW": "EGVW-KAN", "EGVW-KAN no-refit": "EGVW-KAN",
           "EGVW-KAN double-CW": "EGVW-KAN"}
    out = []
    for m in rows_present:
        d = dict(row=m, mean=M[m].mean(), compared_with=ref.get(m, ""))
        r = ref.get(m)
        if r in M.columns:
            d["delta"] = M[m].mean() - M[r].mean()
            try: d["p_wilcoxon"] = stats.wilcoxon(M[m], M[r], zero_method="zsplit").pvalue
            except ValueError: d["p_wilcoxon"] = 1.0
        out.append(d)
    A = pd.DataFrame(out)
    ps = A["p_wilcoxon"].dropna()
    A.loc[ps.index, "p_holm"] = holm(ps.values)
    if "EGVW-KAN" in M.columns and "Ens-uniform" in M.columns:
        try: p_he = stats.wilcoxon(M["EGVW-KAN"], M["Ens-uniform"], zero_method="zsplit").pvalue
        except ValueError: p_he = 1.0
        print(f"  key question - EGVW-KAN vs plain uniform ensemble (B): Δ = "
              f"{M['EGVW-KAN'].mean() - M['Ens-uniform'].mean():+.4f}, Wilcoxon p = {p_he:.4f}")
    A.round(4).to_csv(os.path.join(TAB, f"ablation_{metric}.csv"), index=False)
    print(f"\n== ABLATION ({metric}, {M.shape[0]} datasets) ==\n" + A.round(4).to_string(index=False))
    return A


def sensitivity(R, metric="mcc"):
    S = R[(R.family == "sensitivity") | (R.method == CONTROL)]
    M = S.groupby("method")[metric].mean().sort_values(ascending=False)
    M.round(4).to_csv(os.path.join(TAB, f"sensitivity_{metric}.csv"))
    print(f"\n== SENSITIVITY ({metric}, mean over datasets & folds) ==\n" + M.round(4).to_string())
    g = {1.0: M.get(CONTROL, np.nan)}
    for m, v in M.items():
        if m.startswith("EGVW-KAN [γ="): g[float(m.split("=")[1].rstrip("]"))] = v
    if len(g) > 1:
        xs = sorted(g); fig, ax = plt.subplots(figsize=(4.5, 3))
        ax.plot(xs, [g[x] for x in xs], "o-"); ax.set_xscale("log", base=2)
        ax.set_xlabel("temperature γ"); ax.set_ylabel(f"mean {metric}"); ax.set_title("Sensitivity to γ")
        fig.tight_layout(); fig.savefig(os.path.join(FIG, "sensitivity_gamma.png"), dpi=200); plt.close(fig)


def stability(R, metric="mcc"):
    """spread across repeats (seeds) of the per-repeat dataset mean."""
    sub = R[R.method.isin(["KAN (tuned)", CONTROL])]
    per_rep = sub.groupby(["dataset", "method", "repeat"])[metric].mean()
    S = per_rep.groupby(["dataset", "method"]).std().unstack().round(4)
    S.to_csv(os.path.join(TAB, f"stability_across_repeats_{metric}.csv"))
    print(f"\n== SD ACROSS REPEATS ({metric}) ==\n" + S.to_string() + f"\n  mean: {S.mean().round(4).to_dict()}")


def imbalance_plot(R, bundles_table, metric="mcc"):
    M = dataset_means(R[R.family.isin(["main", "baseline"])], metric)
    base = [c for c in M.columns if c not in MAIN_KAN]
    ir = bundles_table.set_index("dataset")["IR"]
    d1 = M[CONTROL] - M["KAN (tuned)"]; d2 = M[CONTROL] - M[base].max(axis=1)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    x = np.log10(ir.reindex(M.index).astype(float))
    ax.scatter(x, d1, label="vs tuned single KAN"); ax.scatter(x, d2, marker="s", label="vs best classical baseline")
    for ds in M.index: ax.annotate(ds[:18], (x[ds], d2[ds]), fontsize=6)
    ax.axhline(0, color="k", lw=0.8); ax.set_xlabel("log10 imbalance ratio"); ax.set_ylabel(f"Δ{metric} of EGVW-KAN")
    ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(os.path.join(FIG, f"delta_{metric}_vs_ir.png"), dpi=200); plt.close(fig)


def calibration_plots(datasets, methods=("EGVW-KAN", "KAN (tuned)", "Logistic regression", "CatBoost"), bins=10, tag=""):
    for ds in datasets:
        fig, ax = plt.subplots(figsize=(4.2, 4)); ax.plot([0, 1], [0, 1], "k--", lw=0.8)
        for m in methods:
            ys, ps = [], []
            for f in glob.glob(os.path.join(CONFIG["SAVE_DIR"], "preds", f"{_slug(ds)}__r*__f*__*.npz")):
                z = np.load(f, allow_pickle=True); names = list(z["methods"])
                if m in names and z["probs"].ndim == 3 and z["probs"].shape[2] == 2:
                    ys.append(z["y"]); ps.append(z["probs"][names.index(m)][:, 1])
            if not ys: continue
            y, p = np.concatenate(ys), np.concatenate(ps)
            edges = np.linspace(0, 1, bins + 1); idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
            px = [p[idx == b].mean() for b in range(bins) if (idx == b).sum() >= 5]
            py = [y[idx == b].mean() for b in range(bins) if (idx == b).sum() >= 5]
            ax.plot(px, py, "o-", ms=3, label=m)
        ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency"); ax.set_title(ds, fontsize=9)
        ax.legend(fontsize=7); fig.tight_layout(); fig.savefig(os.path.join(FIG, f"calibration{tag}_{_slug(ds)}.png"), dpi=200); plt.close(fig)


def weights_and_guards(I):
    k = I[I.group == "kan"].copy()
    if k.empty: return
    k["weight_entropy"] = k["info"].map(lambda d: d.get("weight_entropy", np.nan) if isinstance(d, dict) else np.nan)
    k["tier"] = k["info"].map(lambda d: d.get("tier") if isinstance(d, dict) else None)
    k["guards"] = k["info"].map(lambda d: ",".join(d.get("guards", [])) if isinstance(d, dict) else "")
    k["failures"] = k["info"].map(lambda d: d.get("egvw_failures", 0) if isinstance(d, dict) else 0)
    W = k.groupby("dataset").agg(tier=("tier", "first"), weight_entropy=("weight_entropy", "mean"),
                                 member_failures=("failures", "sum"))
    W.round(3).to_csv(os.path.join(TAB, "egvw_weight_entropy_tiers.csv"))
    print("\n== EGVW weight entropy (1 = uniform) and tiers ==\n" + W.round(3).to_string())


def timing(R):
    T = R.groupby("method")["time_sec"].median().dropna().sort_values()
    T.round(2).to_csv(os.path.join(TAB, "median_train_time_per_fold.csv"))
    print("\n== median wall time per outer fold incl. tuning (s) ==\n" + T.round(1).to_string())


def check_folds(R):
    """All notebooks must have evaluated the same test rows for each (dataset, repeat, fold)."""
    if "fold_hash" not in R: return
    n = R.groupby(["dataset", "repeat", "fold"]).fold_hash.nunique()
    bad = n[n > 1]
    if len(bad):
        raise RuntimeError(f"Test folds differ between notebooks for {len(bad)} (dataset, repeat, fold) cells, e.g. "
                           f"{list(bad.index[:3])}. Paired statistics would be invalid - the notebooks must use the "
                           f"same CONFIG (BASE_SEED, REPEATS, OUTER_FOLDS, DATASETS) and the same inclusion table.")
    print(f"  fold check passed: {len(n)} (dataset, repeat, fold) cells share identical test rows across notebooks")


# exports: all metrics per fold, error estimates, confusion matrices, per-patient predictions
# All of this is recomputed from results.jsonl and preds/*.npz, so it also works on older runs
# (no retraining).
EXPORT_METRICS = ["accuracy", "balanced_accuracy", "precision", "recall", "specificity", "f1", "mcc",
                  "auroc", "auprc", "brier", "prob_mse", "prob_rmse"]
LOWER_BETTER = {"brier", "prob_mse", "prob_rmse"}
METRIC_RANGE = {"mcc": (-1.0, 1.0), "brier": (0.0, 2.0)}
METRIC_NOTES = {
    "precision": "binary: positive class = clinical condition; K > 2: macro average",
    "recall": "= sensitivity; binary: positive class; K > 2: macro average",
    "specificity": "binary: negative class; K > 2: macro average of one-vs-rest",
    "f1": "binary: positive class; K > 2: macro average",
    "brier": "multiclass Brier score, sum over classes (0 = perfect, 2 = worst)",
    "prob_mse": "mean squared error of the predicted class probabilities against the one-hot outcome = Brier / K; "
                "for binary tasks the MSE of P(positive) against the 0/1 outcome (the binary Brier score)",
    "prob_rmse": "root-mean-squared error of the predicted class probabilities = sqrt(Brier / K); "
                 "for binary tasks the RMSE of P(positive) against the 0/1 outcome",
}


def _k_of(inc):
    return dict(zip(inc.dataset, inc.K))


def _with_derived(R, inc):
    R = R.copy()
    R["recall"] = R["sensitivity"]
    R["prob_mse"] = R["brier"] / R["dataset"].map(_k_of(inc)).astype(float)
    R["prob_rmse"] = np.sqrt(R["prob_mse"])
    return R


def export_per_fold(R, inc):
    """tables/per_fold_metrics.csv, one row per dataset × repeat × outer fold × method, every metric."""
    R = _with_derived(R, inc)
    cols = ["dataset", "repeat", "fold", "method", "family", "n_train", "n_test"] + EXPORT_METRICS + ["time_sec"]
    out = R[[c for c in cols if c in R]].sort_values(["dataset", "method", "repeat", "fold"])
    out.to_csv(os.path.join(TAB, "per_fold_metrics.csv"), index=False, float_format="%.6f")
    pd.DataFrame([{"metric": m, "definition": METRIC_NOTES.get(m, "")} for m in EXPORT_METRICS]) \
        .to_csv(os.path.join(TAB, "metric_definitions.csv"), index=False)
    print(f"\n== per-fold metrics: {len(out)} rows -> per_fold_metrics.csv ==")


def summary_with_error(R, inc):
    """Mean, SD and the measurement error of each estimate. The standard error is the Nadeau–Bengio corrected
    one: outer folds share training rows, so SD/sqrt(J) would understate the uncertainty.
    SE = sqrt((1/J + n_test/n_train) · var), 95 % CI = mean ± t(0.975, J−1) · SE."""
    R = _with_derived(R, inc)
    rows = []
    for (ds, m), g in R.groupby(["dataset", "method"]):
        J = len(g); ratio = g.n_test.mean() / g.n_train.mean()
        for met in EXPORT_METRICS:
            v = g[met].dropna().values; J = len(v)
            if J == 0: continue
            mu = v.mean(); sd = v.std(ddof=1) if J > 1 else np.nan
            se = np.sqrt((1 / J + ratio) * v.var(ddof=1)) if J > 1 else np.nan
            h = stats.t.ppf(0.975, J - 1) * se if J > 1 else np.nan
            lo_b, hi_b = METRIC_RANGE.get(met, (0.0, 1.0))  # keep intervals inside the range of the metric
            rows.append(dict(dataset=ds, method=m, family=g.family.iloc[0], metric=met, n_folds=J, mean=mu, sd=sd,
                             se_naive=sd / np.sqrt(J) if J > 1 else np.nan, se_corrected=se,
                             ci95_low=max(mu - h, lo_b), ci95_high=min(mu + h, hi_b)))
    S = pd.DataFrame(rows)
    S.to_csv(os.path.join(TAB, "summary_by_dataset_all_metrics.csv"), index=False, float_format="%.6f")
    # "mean +- SD" table per metric, without the post-hoc sensitivity rows
    main = S[S.family != "sensitivity"]
    for met in EXPORT_METRICS:
        s = main[main.metric == met]
        if s.empty: continue
        M = s.pivot(index="dataset", columns="method", values="mean")
        D = s.pivot(index="dataset", columns="method", values="sd")
        f3 = lambda x: x.map(lambda v: "" if pd.isna(v) else f"{v:.3f}")
        T = f3(M) + " ± " + f3(D)
        T.loc["Mean over datasets"] = f3(M.mean()) + " ± " + f3(M.std(ddof=1))
        order = M.mean().sort_values(ascending=met in LOWER_BETTER).index
        T[order].to_csv(os.path.join(TAB, f"table_{met}_mean_sd.csv"))
    print("== mean ± SD per metric -> table_<metric>_mean_sd.csv; SE and 95 % CI -> summary_by_dataset_all_metrics.csv ==")
    return S


# display names only; the loaders keep the original codes, so encoding and folds do not change
CLASS_DISPLAY = {"Dermatology": ["psoriasis", "seborrheic dermatitis", "lichen planus", "pityriasis rosea",
                                 "chronic dermatitis", "pityriasis rubra pilaris"]}


def _class_names(ds, K):
    if len(CLASS_DISPLAY.get(ds, [])) == K: return CLASS_DISPLAY[ds]
    b = globals().get("BUNDLES", {}).get(ds)
    return list(b.class_names) if b is not None and len(b.class_names) == K else [f"class {k}" for k in range(K)]


def _read_preds():
    """Yields (dataset slug, repeat, fold, y_true, test-row indices, method names, probs[M, n, K])."""
    import re
    for f in sorted(glob.glob(os.path.join(CONFIG["SAVE_DIR"], "preds", "*.npz"))):
        mt = re.match(r"^(.*)__r(\d+)__f(\d+)__(\w+)\.npz$", os.path.basename(f))
        if not mt: continue
        z = np.load(f, allow_pickle=True)
        if z["probs"].ndim != 3: continue
        yield mt.group(1), int(mt.group(2)), int(mt.group(3)), z["y"], z["te_idx"], list(z["methods"]), z["probs"]


def _plot_confusion(ds, mats, names, path):
    """mats: {method: K×K mean counts per repeat}. Cell colour = row share (recall of that class), so
    classes of very different size are comparable; the text gives the count and the row percentage."""
    from matplotlib.colors import LinearSegmentedColormap
    cmap = LinearSegmentedColormap.from_list("seq_blue", ["#fcfcfb", "#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
    K = len(names); n = len(mats)
    ncol = 2 if (K > 3 and n > 2) else n; nrow = int(np.ceil(n / ncol))  # 2 x 2 grid for many classes
    size = 1.0 + 0.62 * K
    lab_w = 0.09 * max(len(str(x)) for x in names) if K > 3 else 0.4  # space for long class names
    fig, axes = plt.subplots(nrow, ncol, figsize=(size * ncol + lab_w + 0.6, (size + lab_w * 0.7 + 0.5) * nrow + 0.4),
                             squeeze=False)
    for ax in axes.flat[n:]: ax.axis("off")
    for idx, (ax, (m, C)) in enumerate(zip(axes.flat, mats.items())):
        first_col, bottom = idx % ncol == 0, idx // ncol == nrow - 1 or idx + ncol >= n
        share = C / np.maximum(C.sum(1, keepdims=True), 1e-12)
        ax.imshow(share, cmap=cmap, vmin=0, vmax=1)
        for i in range(K):
            for j in range(K):
                cnt = f"{C[i, j]:.0f}" if np.allclose(C[i, j], round(C[i, j])) else f"{C[i, j]:.1f}"
                ax.text(j, i, f"{cnt}\n{100 * share[i, j]:.0f}%", ha="center", va="center",
                        fontsize=7 if K <= 3 else 6, color="#ffffff" if share[i, j] > 0.55 else "#0b0b0b")
        ax.set_xticks(range(K)); ax.set_yticks(range(K))
        if bottom:
            ax.set_xticklabels(names, rotation=45 if K > 3 else 0, ha="right" if K > 3 else "center",
                               rotation_mode="anchor", fontsize=7)
            ax.set_xlabel("predicted", fontsize=8, color="#52514e")
        else: ax.set_xticklabels([])
        if first_col:
            ax.set_yticklabels(names, fontsize=7); ax.set_ylabel("true", fontsize=8, color="#52514e")
        else: ax.set_yticklabels([])
        ax.set_title(m, fontsize=8)
        for s in ax.spines.values(): s.set_visible(False)
        ax.tick_params(length=0)
    fig.suptitle(f"{ds} - pooled over all outer test folds, mean per repeat", fontsize=9)
    fig.tight_layout(); fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.1); plt.close(fig)


def confusion_and_predictions(R, inc, n_plot_extra=1):
    """tables/confusion_matrices_per_fold.csv  every method, every fold (long format)
    predictions/<dataset>.csv                  every test patient, every fold and repeat, every method
    figures/confusion/confusion_<dataset>.png  EGVW-KAN, tuned KAN, best baseline, best KAN variant"""
    names_by_slug = {_slug(d): d for d in inc.dataset}
    Kmap = _k_of(inc)
    PRED = os.path.join(CONFIG["SAVE_DIR"], "predictions"); CONF = os.path.join(FIG, "confusion")
    os.makedirs(PRED, exist_ok=True); os.makedirs(CONF, exist_ok=True)
    cm_rows, pooled, pred_parts = [], {}, {}
    for slug, r, f, y, te, methods, P in _read_preds():
        ds = names_by_slug.get(slug)
        if ds is None: continue
        K = int(Kmap[ds]); labels = list(range(K))
        for i, m in enumerate(methods):
            pred = P[i].argmax(1)
            C = confusion_matrix(y, pred, labels=labels)
            pooled.setdefault((ds, m), {}).setdefault(r, np.zeros((K, K)))
            pooled[(ds, m)][r] += C
            for a in labels:
                for b in labels:
                    cm_rows.append((ds, m, r, f, a, b, int(C[a, b])))
            if "[" in m: continue  # post-hoc gamma / M rows are not written per patient
            part = pd.DataFrame({"dataset": ds, "repeat": r, "fold": f, "row_index": te, "y_true": y,
                                 "method": m, "y_pred": pred})
            for k in labels: part[f"p_{k}"] = P[i][:, k]
            pred_parts.setdefault(ds, []).append(part)
    if not cm_rows:
        print("  no prediction files yet"); return
    CM = pd.DataFrame(cm_rows, columns=["dataset", "method", "repeat", "fold", "true_class", "predicted_class", "count"])
    CM.to_csv(os.path.join(TAB, "confusion_matrices_per_fold.csv"), index=False)
    for ds, parts in pred_parts.items():
        D = pd.concat(parts).sort_values(["method", "repeat", "fold", "row_index"])
        D.to_csv(os.path.join(PRED, f"predictions_{_slug(ds)}.csv"), index=False, float_format="%.6f")
    # figures: the proposed method, the single KAN, and the best model of each family
    fam = R.drop_duplicates("method").set_index("method").family.to_dict()
    for ds in sorted({d for d, _ in pooled}):
        K = int(Kmap[ds]); names = _class_names(ds, K)
        show = [m for m in ("EGVW-KAN", "KAN (tuned)") if (ds, m) in pooled]
        sub = R[R.dataset == ds]
        for family in ("baseline", "kan_variant"):
            best = sub[sub.family == family].groupby("method").mcc.mean().sort_values(ascending=False)
            show += [m for m in best.index[:n_plot_extra] if (ds, m) in pooled]
        if not show: continue
        mats = {}
        for m in show:
            reps = pooled[(ds, m)]
            label = m if fam.get(m) not in ("baseline", "kan_variant") else f"{m}\n(best {'baseline' if fam[m] == 'baseline' else 'KAN variant'} by MCC)"
            mats[label] = sum(reps.values()) / len(reps)
        _plot_confusion(ds, mats, names, os.path.join(CONF, f"confusion_{_slug(ds)}.png"))
        # Part 4: EGVW-KAN next to the resampling rows (same folds and tuned configuration)
        rs = [m for m in ["EGVW-KAN", "EGVW-KAN + SMOTE-NC", "EGVW-KAN + RUS",
                          "EGVW-KAN (no correction) + prior threshold"] if (ds, m) in pooled]
        if len(rs) > 1 and rs[0] == "EGVW-KAN":
            short = {"EGVW-KAN (no correction) + prior threshold": "EGVW-KAN, no correction\n+ prior threshold"}
            _plot_confusion(ds, {short.get(m, m): sum(pooled[(ds, m)].values()) / len(pooled[(ds, m)]) for m in rs}, names,
                            os.path.join(CONF, f"confusion_resampling_{_slug(ds)}.png"))
    print(f"== confusion matrices: {CM[['dataset','method']].drop_duplicates().shape[0]} dataset×method pairs -> "
          f"confusion_matrices_per_fold.csv, figures/confusion/; per-patient predictions -> predictions/ ==")


RESAMPLING_ROWS = ["EGVW-KAN + SMOTE-NC", "EGVW-KAN + RUS", "EGVW-KAN (no correction)",
                   "EGVW-KAN (no correction) + prior threshold"]


def resampling_table(R, inc):
    """Part 4 vs EGVW-KAN, paired on identical folds, identical hyperparameters and member seeds.
    Per metric: mean difference over datasets (row − EGVW-KAN), datasets won/lost, Wilcoxon signed-rank over
    datasets with Holm correction across the four rows. Brier/prob_rmse: lower is better, so a negative
    difference favours the row. The prior-threshold row changes decisions only: its Brier is not reported."""
    rows = [m for m in RESAMPLING_ROWS if m in set(R.method)]
    if CONTROL not in set(R.method) or not rows:
        print("  resampling comparison needs Part 1 (EGVW-KAN) and Part 4 results"); return
    D = _with_derived(complete_only(R, [CONTROL] + rows), inc)
    if D.empty: print("  no dataset complete for EGVW-KAN and all Part 4 rows yet"); return
    out = []
    for met in ["mcc", "accuracy", "balanced_accuracy", "precision", "recall", "specificity", "f1", "auroc",
                "brier", "prob_mse", "prob_rmse"]:
        M = D.groupby(["dataset", "method"])[met].mean().unstack()
        ps, block = [], []
        for m in rows:
            if met in LOWER_BETTER and m.endswith("prior threshold"): continue
            d = (M[m] - M[CONTROL]).dropna()
            better = (d < 0) if met in LOWER_BETTER else (d > 0)
            try: p = stats.wilcoxon(d).pvalue if (d != 0).any() else 1.0
            except ValueError: p = 1.0
            block.append(dict(metric=met, method=m, n_datasets=len(d), mean_diff_vs_EGVW_KAN=d.mean(),
                              datasets_better=int(better.sum()), datasets_worse=int(((d > 0) if met in LOWER_BETTER else (d < 0)).sum()),
                              p_wilcoxon=p))
            ps.append(p)
        for b, ph in zip(block, holm(ps) if ps else []): b["p_holm"] = ph
        out += block
    T = pd.DataFrame(out)
    T.round(4).to_csv(os.path.join(TAB, "resampling_vs_egvw_kan.csv"), index=False)
    M = D.groupby(["dataset", "method"])["mcc"].mean().unstack()[[CONTROL] + rows]
    M.round(4).to_csv(os.path.join(TAB, "resampling_mcc_by_dataset.csv"))
    print(f"\n== RESAMPLING vs EGVW-KAN ({D.dataset.nunique()} datasets; difference = row − EGVW-KAN) ==")
    print(T[T.metric.isin(["mcc", "balanced_accuracy", "auroc", "brier"])].round(4).to_string(index=False))
    binary = [d for d in inc[(inc.decision == "include") & (inc.K == 2)].sort_values("n", ascending=False).dataset
              if d in set(D.dataset)][:3]
    calibration_plots(binary, methods=(CONTROL, "EGVW-KAN + SMOTE-NC", "EGVW-KAN + RUS", "EGVW-KAN (no correction)"),
                      tag="_resampling")
    return T


def analyse():
    merge_previous_runs(verbose=True)  # adds results of the other notebooks if they are attached
    R, I = load_results()
    if R is None:
        print("  no finished units yet - nothing to analyse. (Part 4 needs Part 1's output attached; "
              "otherwise run Stage 3 first.)")
        return
    check_folds(R)
    fam = set(R.family)
    print("  result families present:", sorted(fam))
    if CONFIG.get("SHARD") and not CONFIG["SMOKE"]:
        print(f"  this notebook ran shard {CONFIG['SHARD']}; datasets of the other shard enter the statistics "
              f"only when the other shard's outputs are attached (Add Input) - until then they are listed as incomplete.")
    done_keys = set(open(os.path.join(CONFIG["SAVE_DIR"], "done.txt")).read().split())
    errs = I[I.error.notna() & ~I.key.isin(done_keys)].drop_duplicates("key", keep="last")
    if len(errs):
        print(f"  {len(errs)} units still incomplete (not in the statistics) - see info.jsonl:")
        print(errs[["key", "error"]].head(10).to_string(index=False))
    inc = pd.read_csv(os.path.join(CONFIG["SAVE_DIR"], "dataset_inclusion.csv"))
    # all parts must contain the same datasets (a failed download in one part would show here)
    mine = set(inc[inc.decision == "include"].dataset)
    for d in matching_dirs():
        f = os.path.join(d, "dataset_inclusion.csv")
        if os.path.exists(f):
            other = pd.read_csv(f); theirs = set(other[other.decision == "include"].dataset)
            if theirs != mine:
                print(f"  WARNING: included datasets differ from {d}: only here {sorted(mine - theirs)}, "
                      f"only there {sorted(theirs - mine)}. Re-run Stage 1 of the part that failed to download "
                      f"them; until then those datasets are left out of the paired statistics.")

    def section(name, fn, *args):  # a failing table does not stop the others
        try: fn(*args)
        except Exception as e: print(f"  [{name} skipped: {type(e).__name__}: {e}]")

    # MCC is the primary metric (fixed before training); the other metrics are reported, not used for selection
    for metric in ("mcc", "balanced_accuracy", "auroc", "brier", "f1", "accuracy"):
        section(f"main comparison ({metric})", main_comparison, R, metric)
    section("per-fold export", export_per_fold, R, inc)
    section("mean, SD, standard error, 95 % CI", summary_with_error, R, inc)
    section("confusion matrices and per-patient predictions", confusion_and_predictions, R, inc)
    if "resampling" in fam:
        section("resampling (Part 4)", resampling_table, R, inc)
    if "ablation" in fam:
        section("ablation (mcc)", ablation_table, R, "mcc"); section("ablation (auroc)", ablation_table, R, "auroc")
        section("sensitivity", sensitivity, R, "mcc"); section("stability", stability, R, "mcc")
        section("weights and guards", weights_and_guards, I)
    section("timing", timing, R)
    if {"main", "baseline"} <= fam:
        section("imbalance plot", imbalance_plot, R, inc, "mcc")
    binary = [d for d in inc[(inc.decision == "include") & (inc.K == 2)].sort_values("n", ascending=False).dataset
              if d in set(R.dataset)][:3]  # only datasets with results here (a shard has half of them)
    section("calibration plots", calibration_plots, binary)
    print(f"\n  tables -> {TAB}\n  figures -> {FIG}")

## Cell 10: interpretability check (agreement between members)

In [ ]:
# Cell 10: interpretability check
# Reading an ensemble through its learned functions only makes sense if the members agree.
# For one binary dataset we fit EGVW-KAN on all rows (configuration = most frequent choice over the
# outer folds), compute each member's partial dependence curve of the positive-class logit for every
# numeric feature, and report the mean pairwise Pearson correlation between members.
# Values near 1 support reading the ensemble through its functions; low values mean we
# cannot make that claim for the dataset.
import collections, json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def member_agreement(bundle, n_grid=25, top=4, seed=CONFIG["BASE_SEED"]):
    assert bundle.K == 2, "use a binary dataset"
    _torch_single_thread()
    I = pd.read_json(os.path.join(CONFIG["SAVE_DIR"], "info.jsonl"), lines=True)
    ps = [json.dumps(d.get("params", {}), sort_keys=True) for d in I[(I.dataset == bundle.name) & (I.group == "kan")]["info"]
          if isinstance(d, dict) and d.get("params")]
    p = json.loads(collections.Counter(ps).most_common(1)[0][0]) if ps else {k: v[0] for k, v in CONFIG["KAN_GRID"].items()}
    p["_K"] = 2
    pre = Preprocessor(bundle.nominal).fit(bundle.X); X = pre.transform(bundle.X); y = bundle.y
    E = egvw_fit(X, y, bundle.groups, p, 2, seed)
    w = member_weights(E["val_loss"], E["val_disp"], "egvw", CONFIG["GAMMA"])
    rows, curves = [], {}
    for j, name in enumerate(pre.num):  # numeric features are the first columns
        grid = np.quantile(X[:, j], np.linspace(0.05, 0.95, n_grid))
        pd_m = []
        for mod in E["refit_models"]:
            vals = []
            for g in grid:
                Xg = X.copy(); Xg[:, j] = g
                z = kan_logits(mod, Xg, 2); vals.append(float((z[:, 1] - z[:, 0]).mean()))
            pd_m.append(np.array(vals))
        P = np.stack(pd_m); curves[name] = (grid, P)
        r = [np.corrcoef(P[a], P[b])[0, 1] for a in range(len(P)) for b in range(a + 1, len(P))
             if P[a].std() > 0 and P[b].std() > 0]
        ens = (w[:, None] * P).sum(0)
        rows.append(dict(feature=name, ensemble_effect_range=float(ens.max() - ens.min()),
                         member_agreement_r=float(np.mean(r)) if r else np.nan))
    A = pd.DataFrame(rows).sort_values("ensemble_effect_range", ascending=False)
    A.round(3).to_csv(os.path.join(TAB, f"member_agreement_{_slug(bundle.name)}.csv"), index=False)
    feats = A.feature.head(top).tolist()
    fig, axes = plt.subplots(1, len(feats), figsize=(3.2 * len(feats), 2.8))
    for ax, f in zip(np.atleast_1d(axes), feats):
        g, P = curves[f]
        for i, c in enumerate(P): ax.plot(g, c, lw=1, alpha=0.7, label=f"member {i+1} (w={w[i]:.2f})")
        ax.plot(g, (w[:, None] * P).sum(0), "k", lw=2, label="EGVW-KAN")
        r = A.set_index("feature").loc[f, "member_agreement_r"]
        ax.set_title(f"{f}\nagreement r = {r:.2f}", fontsize=8); ax.set_xlabel("standardised value", fontsize=7)
    np.atleast_1d(axes)[0].set_ylabel("Δ logit (positive class)", fontsize=7); np.atleast_1d(axes)[0].legend(fontsize=5)
    fig.tight_layout(); fig.savefig(os.path.join(FIG, f"member_agreement_{_slug(bundle.name)}.png"), dpi=200); plt.close(fig)
    print(A.round(3).to_string(index=False))
    return A

---
# Stage 1: data audit
Loads every candidate, applies the corrections and rules R1-R5, and writes `dataset_inclusion.csv`. Check this table before training: every exclusion should name a rule. A load error (`R5/R3`) for a dataset you expected usually means the source or a column name changed. Fix the loader in Cell 3, in all parts.

In [ ]:
BUNDLES, INCLUSION = audit_datasets()

# Stage 2: leakage self-test
Runs one outer fold of the smallest dataset on which every component is active (moderate or extreme imbalance, so class weights and the bootstrap are on), with a small search budget. It tests the code paths, not accuracy.

- Sentinel test: every test-fold label is replaced by an impossible class. Any step that reads a test label then crashes or produces an extra output column.
- Permutation test: the test labels are permuted and the predictions must stay the same. If they change, the normal run is repeated to separate leakage (repeat identical) from numerical noise of the machine (repeat differs).

The cell has to print `PASSED`. It runs once; later sessions reuse the result.

In [ ]:
_marker = os.path.join(CONFIG['SAVE_DIR'], f'leakage_selftest_passed_t2-kan-a_{CONFIG["METHOD_VERSION"]}_{CONFIG["VARIANT_VERSION"]}_{CONFIG["V2_VERSION"]}.txt')
_prev = [os.path.join(d, f'leakage_selftest_passed_t2-kan-a_{CONFIG["METHOD_VERSION"]}_{CONFIG["VARIANT_VERSION"]}_{CONFIG["V2_VERSION"]}.txt') for d in matching_dirs() if os.path.exists(os.path.join(d, f'leakage_selftest_passed_t2-kan-a_{CONFIG["METHOD_VERSION"]}_{CONFIG["VARIANT_VERSION"]}_{CONFIG["V2_VERSION"]}.txt'))]
if os.path.exists(_marker) or _prev:
    print('Leakage self-test already passed in an earlier session:', open((_prev or [_marker])[0]).read())
    if _prev and not os.path.exists(_marker): shutil.copy(_prev[0], _marker)
else:
    _small = selftest_dataset(BUNDLES)
    print('dataset:', _small)
    _lt = leakage_self_test(BUNDLES[_small])
    assert _lt['ok'], 'Leakage self-test failed - do not run the benchmark'
    open(_marker, 'w').write(f"PASSED on {_small}, {time.ctime()}: {_lt['summary']}")

# Stage 3: run this part
Resumable. Prints progress every few minutes.

In [ ]:
run_all(BUNDLES)

# Stage 4: quick look
`analyse_v2()` compares the v2 rows with EGVW-KAN v1 and Ens+CW in this notebook's output (KAN notebooks only). The pre-registered Stage 2 analysis runs locally on the merged outputs of all three notebooks (`analysis/analyse_stage2.py`).

In [ ]:
analyse_v2()